# Extent, distribution, longitudinal persistence, cohort differences, and bulbar associations

This notebook is the frozen production implementation for the cohort-characterization analysis of the six recording-quality families. It retains repeated recordings as distinct observations, models participant clustering explicitly, preserves feature-specific zero and missing-value semantics, and never constructs a family-level or global quality score.

The notebook is intentionally verbose. Every model route, support gate, fallback, transformation, inferential family, sample size, and figure source table is exported for review.


## 0. Frozen statistical specification

**Primary observational unit.** The recording is the primary observational unit. Recordings collected months apart are not collapsed because disease state, acquisition setup, behavior, and environment may change across sessions.

**Clustering.** Continuous outcomes are analyzed with participant-random-intercept mixed models. If the random-effect variance is on the boundary or the model is singular, the prespecified fallback is participant-clustered Gaussian GEE. Binary availability and presence outcomes use participant-clustered binomial GEE.

**Feature routing.** Continuous, one-sided zero-mass, sparse-event, censored, exploratory, monitoring, and audit outputs are handled separately according to the frozen feature registry. Missing values remain missing. A measured zero retains its registered meaning.

**Persistence.** Participant-level persistence is the variance proportion attributable to between-participant differences in an intercept-only mixed model. It is reported as an ICC and is not called technical repeatability. Confidence intervals use participant-cluster bootstrap resampling.

**Longitudinal change.** Within ALS, time is years since each participant's first eligible recording. Random slopes are attempted only when support and model stability permit; otherwise a random-intercept model is used. Longitudinal change is not interpreted as purely technical drift.

**Cohort effects.** ALS-control differences are estimated from clustered recording-level models. Age- and sex-adjusted, participant-collapsed, one-recording-per-participant, untransformed, and winsorized analyses are sensitivities, not replacements for the primary model.

**Bulbar effects.** Recording-matched ALSFRS-R bulbar scores are decomposed into participant mean and recording-specific deviation, separating between-participant differences from within-participant change. Time-adjusted models are sensitivities.

**Multiplicity.** Benjamini-Hochberg correction is applied within prespecified inferential families and separately for core versus secondary/exploratory outputs. Descriptive summaries, ICCs, variance components, and model diagnostics are not FDR-adjusted.

**Figures.** The main figure uses interval and forest geometry rather than density plots. Exact p-values, q-values, model types, and support counts remain in source tables rather than being printed inside the figure. Layout is programmatically audited for text overlap.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable, Independence
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

warnings.filterwarnings('default')
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

# ------------------------------------------------------------------
# Run configuration
# ------------------------------------------------------------------
FINAL_RUN = True
SEED = 20260805
RUN_ICC_BOOTSTRAP = FINAL_RUN
N_ICC_BOOTSTRAP = 2000 if FINAL_RUN else 0
N_ONE_RECORDING_RESAMPLES = 1000 if FINAL_RUN else 100
MIN_BINARY_SMALLER_STATE_PARTICIPANTS = 10
MIN_SPARSE_POSITIVE_RECORDINGS = 20
MIN_SPARSE_POSITIVE_PARTICIPANTS = 10
MIN_REPEATED_PARTICIPANTS = 20
MIN_REPEATED_JOINT_OBSERVATIONS = 50
MIN_BULBAR_REPEATED_PARTICIPANTS = 30
MIN_BULBAR_MATCHED_RECORDINGS = 50
MIN_RANDOM_SLOPE_PARTICIPANTS = 30
MIN_RANDOM_SLOPE_THREE_RECORDINGS = 30
WINSOR_LIMITS = (0.01, 0.99)

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()
DEFAULT_PAPER_ROOT = ANALYSIS_ROOT.parent
PAPER_ROOT = Path(os.environ.get("PAPER1_ROOT", DEFAULT_PAPER_ROOT))
PIPELINE_ROOT = Path(os.environ.get("PAPER1_PIPELINE_ROOT", PAPER_ROOT / "Paper_1_Final_Cleanup"))

DATA_FREEZE_DIR = Path(os.environ.get(
    'PAPER1_DATA_FREEZE_DIR',
    PIPELINE_ROOT / 'outputs' / 'data',
))
FAMILY_FREEZE_DIR = Path(os.environ.get(
    'PAPER1_FAMILY_FREEZE_DIR',
    PIPELINE_ROOT / 'outputs' / 'features',
))

OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'extent_distribution_persistence'
TABLE_DIR = OUTPUT_ROOT / 'tables'
MODEL_DIR = OUTPUT_ROOT / 'models'
LOG_DIR = OUTPUT_ROOT / 'logs'
MANIFEST_DIR = OUTPUT_ROOT / 'manifests'
FIGURE_MAIN_DIR = OUTPUT_ROOT / 'figures' / 'main'
FIGURE_SUPP_DIR = OUTPUT_ROOT / 'figures' / 'supplementary'
FIGURE_SOURCE_DIR = OUTPUT_ROOT / 'figures' / 'source_data'
BOOTSTRAP_DIR = MODEL_DIR / 'icc_bootstrap_checkpoints'

for directory in [
    TABLE_DIR, MODEL_DIR, LOG_DIR, MANIFEST_DIR,
    FIGURE_MAIN_DIR, FIGURE_SUPP_DIR, FIGURE_SOURCE_DIR,
    BOOTSTRAP_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

FROZEN_SPEC = {
    'analysis_name': 'Extent, distribution, longitudinal persistence, cohort and bulbar structure',
    'primary_unit': 'recording',
    'cluster_unit': 'participant_id',
    'continuous_primary_model': 'participant random-intercept mixed model',
    'continuous_fallback': 'participant-clustered Gaussian GEE',
    'binary_model': 'participant-clustered binomial GEE',
    'icc_ci': 'participant-cluster bootstrap',
    'final_run': FINAL_RUN,
    'seed': SEED,
    'run_icc_bootstrap': RUN_ICC_BOOTSTRAP,
    'n_icc_bootstrap': N_ICC_BOOTSTRAP,
    'n_one_recording_resamples': N_ONE_RECORDING_RESAMPLES,
    'winsor_limits': WINSOR_LIMITS,
    'no_family_or_global_score': True,
}
(MANIFEST_DIR / 'frozen_analysis_spec.json').write_text(
    json.dumps(FROZEN_SPEC, indent=2), encoding='utf-8'
)

print(json.dumps(FROZEN_SPEC, indent=2))
print(f'DATA_FREEZE_DIR:   {DATA_FREEZE_DIR}')
print(f'FAMILY_FREEZE_DIR: {FAMILY_FREEZE_DIR}')
print(f'OUTPUT_ROOT:       {OUTPUT_ROOT}')


## 1. Shared utilities


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, low_memory=False)
    if suffix == '.parquet':
        try:
            return pd.read_parquet(path)
        except ImportError:
            csv_fallback = path.with_suffix('.csv')
            if csv_fallback.exists():
                warnings.warn(
                    f'Parquet engine unavailable; using CSV fallback: {csv_fallback}',
                    RuntimeWarning,
                )
                return pd.read_csv(csv_fallback, low_memory=False)
            raise
    raise ValueError(f'Unsupported table type: {path}')


def resolve_table(
    root: Path,
    exact_relatives: Sequence[str],
    patterns: Sequence[str],
    *,
    label: str,
) -> Path:
    for relative in exact_relatives:
        candidate = root / relative
        if candidate.exists():
            return candidate

    matches: list[Path] = []
    for pattern in patterns:
        matches.extend(path for path in root.rglob(pattern) if path.is_file())

    matches = sorted(
        {path.resolve() for path in matches},
        key=lambda path: (
            0 if path.suffix.lower() == '.parquet' else 1,
            'source_candidate' in str(path).lower(),
            len(str(path)),
            str(path).lower(),
        ),
    )
    preferred = [
        path for path in matches
        if 'checkpoint' not in str(path).lower()
    ]
    if preferred:
        matches = preferred
    if not matches:
        raise FileNotFoundError(
            f'Could not locate {label} below {root}. Patterns: {patterns}'
        )
    return matches[0]


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')


def normalize_id(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip()


def transform_feature(series: pd.Series, transform: str) -> pd.Series:
    x = safe_numeric(series)
    if transform == 'none':
        return x
    if transform == 'log1p':
        if (x.dropna() < 0).any():
            raise ValueError('log1p requested for a feature containing negative values.')
        return np.log1p(x)
    if transform == 'asinh':
        return np.arcsinh(x)
    raise ValueError(f'Unknown transform: {transform}')


def robust_scale(
    series: pd.Series,
    *,
    median: float | None = None,
    iqr: float | None = None,
) -> tuple[pd.Series, float, float]:
    numeric = safe_numeric(series)
    if median is None:
        median = float(numeric.median(skipna=True))
    if iqr is None:
        q1, q3 = numeric.quantile([0.25, 0.75])
        iqr = float(q3 - q1)
    if not np.isfinite(iqr) or iqr <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float), median, iqr
    return (numeric - median) / iqr, median, iqr


def bh_adjust(frame: pd.DataFrame, p_col: str = 'p_value') -> pd.DataFrame:
    output = frame.copy()
    output['q_value'] = np.nan
    mask = output[p_col].notna() & np.isfinite(output[p_col])
    if mask.any():
        output.loc[mask, 'q_value'] = multipletests(
            output.loc[mask, p_col].astype(float), method='fdr_bh'
        )[1]
    return output


def percentile_interval(values: Iterable[float], alpha: float = 0.05) -> tuple[float, float]:
    clean = np.asarray([value for value in values if np.isfinite(value)], dtype=float)
    if clean.size == 0:
        return np.nan, np.nan
    return tuple(np.quantile(clean, [alpha / 2, 1 - alpha / 2]))


def export_dataframe(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def save_figure(fig: mpl.figure.Figure, stem: str, *, main: bool) -> dict[str, str]:
    """Export deterministic fixed-canvas publication graphics."""
    directory = FIGURE_MAIN_DIR if main else FIGURE_SUPP_DIR
    paths = {
        'png': directory / f'{stem}.png',
        'svg': directory / f'{stem}.svg',
    }
    # The editable SVG is the vector deliverable for all figures. The main figure
    # additionally receives a PDF. Supplementary PNGs use 350 dpi to keep the
    # complete production run practical without affecting vector quality.
    fig.savefig(paths['png'], dpi=600 if main else 350, facecolor='white')
    fig.savefig(paths['svg'], facecolor='white')
    if main:
        paths['pdf'] = directory / f'{stem}.pdf'
        fig.savefig(paths['pdf'], facecolor='white')
    return {key: str(value) for key, value in paths.items()}


def model_row(
    *, feature: str, analysis_name: str, target_term: str,
    estimate: float, standard_error: float, ci95_low: float, ci95_high: float,
    p_value: float, n_recordings: int, n_participants: int,
    model: str, converged: bool, random_intercept_variance: float = np.nan,
    notes: str = '',
) -> dict[str, Any]:
    return {
        'feature': feature,
        'analysis': analysis_name,
        'target_term': target_term,
        'estimate': estimate,
        'standard_error': standard_error,
        'ci95_low': ci95_low,
        'ci95_high': ci95_high,
        'p_value': p_value,
        'n_recordings': n_recordings,
        'n_participants': n_participants,
        'model': model,
        'converged': bool(converged),
        'random_intercept_variance': random_intercept_variance,
        'notes': notes,
    }


def bowley_skewness(values: pd.Series) -> float:
    clean = safe_numeric(values).dropna()
    if clean.empty:
        return np.nan
    q1, q2, q3 = clean.quantile([0.25, 0.50, 0.75])
    denominator = q3 - q1
    if not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return float((q3 + q1 - 2 * q2) / denominator)


def boolean_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return (
        series.astype('string').str.strip().str.lower()
        .map({'true': True, 'false': False, '1': True, '0': False,
              'yes': True, 'no': False})
        .fillna(False)
        .astype(bool)
    )


## 2. Discover and hash immutable inputs


In [ ]:
metadata_path = resolve_table(
    DATA_FREEZE_DIR,
    exact_relatives=['bamboo_recording_freeze_ledger.csv', 'bamboo_recording_freeze_ledger.parquet'],
    patterns=['*recording*freeze*ledger*.parquet', '*recording*freeze*ledger*.csv'],
    label='recording freeze ledger',
)

family_paths = {
    'QADD': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'families\02_QADD\features.parquet',
            r'additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.parquet',
            r'additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.csv',
        ],
        patterns=['qadd_v420_recording_features.parquet', 'qadd_v420_recording_features.csv'],
        label='QADD recording features',
    ),
    'QGAIN': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'families\01_QGAIN\features.parquet',
            r'gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.parquet',
            r'gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.csv',
        ],
        patterns=['qgain_v410_recording_features.parquet', 'qgain_v410_recording_features.csv'],
        label='QGAIN recording features',
    ),
    'QREV': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'families\03_QREV\features.parquet',
            r'reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.parquet',
            r'reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.csv',
        ],
        patterns=['qrev_v400_analysis_features.parquet', 'qrev_v400_analysis_features.csv'],
        label='QREV analysis features',
    ),
    'QCHAN': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'families\04_QCHAN\features.parquet',
            r'channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.parquet',
            r'channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.csv',
        ],
        patterns=['qchan_v400_analysis_features.parquet', 'qchan_v400_analysis_features.csv'],
        label='QCHAN analysis features',
    ),
    'QDIST': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'families\05_QDIST\features.parquet',
            r'nonlinear_distortion\qdist-v4.1.0\tables\qdist_v410_recording_features.parquet',
            r'nonlinear_distortion\qdist-v4.1.0\tables\qdist_v410_recording_features.csv',
        ],
        patterns=['qdist_v410_recording_features.parquet', 'qdist_v410_recording_features.csv'],
        label='QDIST recording features',
    ),
    'QTEMP': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'experimental\QTEMP_LEGACY_NO_RETAINED\qtemp_v100_exploratory_features.csv',
            r'temporal_discontinuity\qtemp-v1.0.0-analytical-final-no-retained\tables\qtemp_v100_exploratory_features.csv',
        ],
        patterns=['qtemp_v100_exploratory_features.csv'],
        label='QTEMP exploratory features',
    ),
}

input_inventory = pd.DataFrame([
    {
        'input': 'metadata', 'path': str(metadata_path),
        'sha256': sha256_file(metadata_path), 'bytes': metadata_path.stat().st_size,
    },
    *[
        {
            'input': family, 'path': str(path),
            'sha256': sha256_file(path), 'bytes': path.stat().st_size,
        }
        for family, path in family_paths.items()
    ],
])
display(input_inventory)
export_dataframe(input_inventory, MANIFEST_DIR / 'input_inventory.csv')

metadata_raw = read_table(metadata_path)
family_raw = {family: read_table(path) for family, path in family_paths.items()}

print('Metadata:', metadata_raw.shape)
for family, frame in family_raw.items():
    print(f'{family}: {frame.shape}')


## 3. Frozen feature registry and model routing


In [ ]:
FEATURE_REGISTRY = pd.DataFrame([
    # QADD
    dict(feature='qadd_pause_ac_level_dbfs_median', family='QADD', label='Pause-region level',
         short_label='Pause-region level', unit='dBFS', validated_role='primary_contextual',
         inference_tier='core', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_pause_level_support_tier',
         exposure_field=None, direction='higher recorded pause-region level; contextual',
         zero_semantics='digital full scale reference; zero is not a cleanliness threshold',
         include_main=True, include_persistence=True),
    dict(feature='qadd_pause_level_iqr_db', family='QADD', label='Pause-level variability',
         short_label='Pause-level IQR', unit='dB', validated_role='secondary',
         inference_tier='secondary', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_pause_dispersion_support_tier',
         exposure_field=None, direction='higher pause-level variability',
         zero_semantics='measured zero variability where available',
         include_main=True, include_persistence=True),
    dict(feature='qadd_speech_pause_level_contrast_db', family='QADD', label='Speech-pause level contrast',
         short_label='Speech-pause contrast', unit='dB', validated_role='secondary_mixed',
         inference_tier='secondary', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_speech_pause_contrast_support_tier',
         exposure_field=None, direction='higher speech-pause separation; not an adverse-quality direction',
         zero_semantics='equal speech and pause levels in the defined view',
         include_main=True, include_persistence=True),
    dict(feature='qadd_pause_spectral_flatness', family='QADD', label='Pause spectral flatness',
         short_label='Pause spectral flatness', unit='ratio', validated_role='secondary_nonordinal',
         inference_tier='secondary', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_flatness_support_tier',
         exposure_field=None, direction='nonordinal spectral structure',
         zero_semantics='structured/tonal extreme; not clean',
         include_main=True, include_persistence=True),
    dict(feature='qadd_mains_hum_comb_score_db', family='QADD', label='Hum-comb prominence',
         short_label='Hum-comb prominence', unit='dB', validated_role='targeted_conditional',
         inference_tier='secondary', transform='asinh', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_hum_support_tier',
         exposure_field=None, direction='higher hum-pattern prominence',
         zero_semantics='no excess on this score; not proof of no hum',
         include_main=True, include_persistence=True),

    # QGAIN
    dict(feature='qgain_typical_speech_level_dbfs', family='QGAIN', label='Typical speech level',
         short_label='Typical speech level', unit='dBFS', validated_role='primary_contextual',
         inference_tier='core', transform='none', outcome_class='continuous',
         availability_field='qgain_typical_speech_level_dbfs_available', censor_field=None,
         support_field='qgain_typical_speech_level_dbfs_support_tier', exposure_field=None,
         direction='higher digital level (less negative dBFS); contextual',
         zero_semantics='digital full scale reference; not calibrated SPL',
         include_main=True, include_persistence=True),
    dict(feature='qgain_within_segment_iqr_db', family='QGAIN', label='Within-segment level IQR',
         short_label='Within-segment IQR', unit='dB', validated_role='primary_mixed',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field='qgain_within_segment_iqr_db_available', censor_field=None,
         support_field='qgain_within_segment_iqr_db_support_tier', exposure_field=None,
         direction='higher within-segment level variability',
         zero_semantics='no measured within-segment dispersion',
         include_main=True, include_persistence=True),
    dict(feature='qgain_between_segment_mad_db', family='QGAIN', label='Between-segment level MAD',
         short_label='Between-segment MAD', unit='dB', validated_role='secondary_mixed',
         inference_tier='secondary', transform='log1p', outcome_class='continuous',
         availability_field='qgain_between_segment_mad_db_available', censor_field=None,
         support_field='qgain_between_segment_mad_db_support_tier', exposure_field=None,
         direction='higher between-segment level variability',
         zero_semantics='no measured between-segment dispersion',
         include_main=True, include_persistence=True),
    dict(feature='qgain_abs_drift_db_per_min', family='QGAIN', label='Absolute level drift',
         short_label='Absolute level drift', unit='dB/min', validated_role='exploratory_contextual',
         inference_tier='exploratory', transform='log1p', outcome_class='continuous',
         availability_field='qgain_abs_drift_db_per_min_available', censor_field=None,
         support_field='qgain_abs_drift_db_per_min_support_tier', exposure_field=None,
         direction='higher absolute monotonic level drift',
         zero_semantics='no estimated monotonic drift',
         include_main=False, include_persistence=True),

    # QREV
    dict(feature='qrev_tail_excess_100ms_db', family='QREV', label='Early residual-tail excess',
         short_label='Early tail excess†', unit='dB', validated_role='primary_conditional',
         inference_tier='core', transform='asinh', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_tail_excess_100ms_db_support_tier',
         exposure_field=None, direction='higher early residual energy',
         zero_semantics='early and late regions equal on the defined scale',
         include_main=True, include_persistence=True),
    dict(feature='qrev_tail_persistence_median_sec', family='QREV', label='Tail persistence',
         short_label='Tail persistence†', unit='s', validated_role='secondary_conditional_nonindependent',
         inference_tier='secondary', transform='log1p', outcome_class='censored_continuous',
         availability_field=None, censor_field='qrev_persistence_recording_median_censored',
         support_field='qrev_tail_persistence_median_sec_support_tier', exposure_field=None,
         direction='higher residual persistence',
         zero_semantics='immediate return within estimator resolution',
         include_main=True, include_persistence=True),
    dict(feature='qrev_downward_decay_rate_db_per_sec', family='QREV', label='Downward residual decay rate',
         short_label='Downward decay rate†', unit='dB/s', validated_role='exploratory_conditional',
         inference_tier='exploratory', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_downward_decay_rate_db_per_sec_support_tier',
         exposure_field=None, direction='higher means faster downward decay, not more reverberation',
         zero_semantics='conditional on a valid decline',
         include_main=False, include_persistence=True),
    dict(feature='qrev_srmr_norm', family='QREV', label='Normalized-fast SRMR',
         short_label='Normalized-fast SRMR', unit='ratio', validated_role='established_comparator',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_srmr_norm_support_tier',
         exposure_field=None, direction='higher generally indicates less modulation smearing',
         zero_semantics='theoretical lower bound; not a clean/unclean state',
         include_main=True, include_persistence=True),

    # QCHAN
    dict(feature='qchan_ltas_distance_db', family='QCHAN', label='LTAS distance',
         short_label='LTAS distance', unit='dB RMS', validated_role='primary_nonordinal',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qchan_ltas_distance_db_support_tier',
         exposure_field=None, direction='higher reference-relative spectral deviation; nonordinal',
         zero_semantics='exact match to the reference spectral shape',
         include_main=True, include_persistence=True),
    dict(feature='qchan_rolloff95_deficit_hz', family='QCHAN', label='Rolloff-95 deficit',
         short_label='Rolloff-95 deficit', unit='Hz', validated_role='primary_one_sided',
         inference_tier='core', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_rolloff95_deficit_hz_support_tier',
         exposure_field=None, direction='higher upper-frequency deficit',
         zero_semantics='target rolloff equal to or above the reference',
         include_main=True, include_persistence=True),
    dict(feature='qchan_highband_ratio_deficit', family='QCHAN', label='High-band ratio deficit',
         short_label='High-band ratio deficit', unit='proportion', validated_role='secondary_nonindependent',
         inference_tier='secondary', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_highband_ratio_deficit_support_tier',
         exposure_field=None, direction='higher high-band deficit',
         zero_semantics='target high-band share equal to or above the reference',
         include_main=True, include_persistence=True),
    dict(feature='qchan_tilt_steepening_db_per_oct', family='QCHAN', label='Spectral-tilt steepening',
         short_label='Spectral-tilt steepening', unit='dB/oct', validated_role='exploratory_phenotype_sensitive',
         inference_tier='exploratory', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_tilt_steepening_db_per_oct_support_tier',
         exposure_field=None, direction='higher reference-relative tilt steepening',
         zero_semantics='target tilt equal to or shallower than the reference',
         include_main=False, include_persistence=True),

    # QDIST
    dict(feature='qdist_hard_clipped_sample_fraction', family='QDIST', label='Hard-clipped sample fraction',
         short_label='Hard-clipping events', unit='fraction', validated_role='primary_sparse_burden',
         inference_tier='core', transform='log1p', outcome_class='sparse_event_extent',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='higher detected hard-plateau burden',
         zero_semantics='no detector-defined hard-plateau support observed',
         include_main=True, include_persistence=False),
    dict(feature='qdist_hard_clip_event_rate_per_min', family='QDIST', label='Hard-clipping episode rate',
         short_label='Hard-clipping episode rate', unit='episodes/min', validated_role='secondary_sparse_event',
         inference_tier='secondary', transform='log1p', outcome_class='sparse_event_rate',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='higher detected episode frequency',
         zero_semantics='no detector-defined episode observed',
         include_main=False, include_persistence=False),
    dict(feature='qdist_hard_clipped_frame_fraction', family='QDIST', label='Hard-clipped frame fraction',
         short_label='Hard-clipped frame fraction', unit='fraction', validated_role='conditional_audit',
         inference_tier='audit', transform='log1p', outcome_class='audit',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='audit only',
         zero_semantics='no grid frame intersected an accepted plateau',
         include_main=False, include_persistence=False),

    # QTEMP
    dict(feature='qtemp_dropout_duration_fraction', family='QTEMP', label='Dropout-like duration fraction',
         short_label='Dropout-like events', unit='fraction', validated_role='exploratory_not_primary',
         inference_tier='exploratory', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='exploratory detected burden', zero_semantics='no event met the registered dropout-like rule',
         include_main=True, include_persistence=False),
    dict(feature='qtemp_dropout_event_rate_per_min', family='QTEMP', label='Dropout-like event rate',
         short_label='Dropout-like event rate', unit='events/min', validated_role='exploratory_not_primary',
         inference_tier='exploratory', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='exploratory detected frequency', zero_semantics='no event met the registered dropout-like rule',
         include_main=False, include_persistence=False),
    dict(feature='qtemp_frozen_audio_duration_fraction', family='QTEMP', label='Frozen-audio duration fraction',
         short_label='Frozen-audio events', unit='fraction', validated_role='monitoring_zero_variation',
         inference_tier='monitoring', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='monitoring only', zero_semantics='no event met the registered repetition rule',
         include_main=True, include_persistence=False),
    dict(feature='qtemp_frozen_audio_event_rate_per_min', family='QTEMP', label='Frozen-audio event rate',
         short_label='Frozen-audio event rate', unit='events/min', validated_role='monitoring_zero_variation',
         inference_tier='monitoring', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='monitoring only', zero_semantics='no event met the registered repetition rule',
         include_main=False, include_persistence=False),
])
FEATURE_REGISTRY['feature_order'] = np.arange(len(FEATURE_REGISTRY))
FEATURE_REGISTRY['is_core'] = FEATURE_REGISTRY['inference_tier'].eq('core')

required_registry_fields = [
    'feature', 'family', 'label', 'short_label', 'unit', 'validated_role',
    'inference_tier', 'transform', 'outcome_class', 'availability_field',
    'censor_field', 'support_field', 'exposure_field', 'direction',
    'zero_semantics', 'include_main', 'include_persistence', 'feature_order',
]
assert not FEATURE_REGISTRY[required_registry_fields].isna().all(axis=1).any()
display(FEATURE_REGISTRY)
export_dataframe(FEATURE_REGISTRY, TABLE_DIR / 'feature_registry.csv')


## 4. Harmonize family tables and construct the common analysis matrix


In [ ]:
def select_family_columns(frame: pd.DataFrame, family_prefix: str) -> pd.DataFrame:
    if 'logical_recording_id' not in frame.columns:
        raise KeyError(f'{family_prefix}: logical_recording_id is missing.')
    columns = ['logical_recording_id'] + [
        column for column in frame.columns
        if column.lower().startswith(family_prefix.lower())
    ]
    selected = frame.loc[:, list(dict.fromkeys(columns))].copy()
    selected['logical_recording_id'] = normalize_id(selected['logical_recording_id'])
    if selected['logical_recording_id'].duplicated().any():
        examples = selected.loc[
            selected['logical_recording_id'].duplicated(keep=False),
            'logical_recording_id',
        ].head(10).tolist()
        raise AssertionError(f'{family_prefix}: duplicated recording IDs: {examples}')
    return selected

family_tables = {
    family: select_family_columns(frame, family.lower())
    for family, frame in family_raw.items()
}

id_sets = {
    family: set(frame['logical_recording_id'].dropna())
    for family, frame in family_tables.items()
}
intersection_ids = set.intersection(*id_sets.values())
union_ids = set.union(*id_sets.values())

family_id_summary = pd.DataFrame([
    {
        'family': family,
        'rows': len(frame),
        'unique_recordings': frame['logical_recording_id'].nunique(),
        'missing_from_union': len(union_ids - ids),
    }
    for (family, frame), ids in zip(family_tables.items(), id_sets.values())
])
display(family_id_summary)

if len(intersection_ids) != 519 or len(union_ids) != 519:
    discrepancy = []
    for recording_id in sorted(union_ids):
        missing = [family for family, ids in id_sets.items() if recording_id not in ids]
        if missing:
            discrepancy.append({
                'logical_recording_id': recording_id,
                'missing_families': '|'.join(missing),
            })
    discrepancy_frame = pd.DataFrame(discrepancy)
    export_dataframe(discrepancy_frame, LOG_DIR / 'family_recording_id_discrepancies.csv')
    raise AssertionError('The six family tables do not define one common 519-recording population.')

feature_matrix = pd.DataFrame({'logical_recording_id': sorted(intersection_ids)})
for family in ['QADD', 'QGAIN', 'QREV', 'QCHAN', 'QDIST', 'QTEMP']:
    feature_matrix = feature_matrix.merge(
        family_tables[family], on='logical_recording_id', how='left', validate='one_to_one'
    )

missing_features = [
    feature for feature in FEATURE_REGISTRY['feature']
    if feature not in feature_matrix.columns
]
if missing_features:
    raise KeyError(f'Required feature columns are missing: {missing_features}')

metadata = metadata_raw.copy()
if 'logical_recording_id' not in metadata.columns:
    raise KeyError('Metadata ledger does not contain logical_recording_id.')
metadata['logical_recording_id'] = normalize_id(metadata['logical_recording_id'])
if metadata['logical_recording_id'].duplicated().any():
    raise AssertionError('Metadata contains duplicated logical_recording_id values.')
metadata = metadata.loc[metadata['logical_recording_id'].isin(intersection_ids)].copy()

participant_source = 'SubjectID' if 'SubjectID' in metadata.columns else 'subject'
metadata['participant_id'] = normalize_id(metadata[participant_source])
metadata['cohort'] = (
    metadata['diagnosis_analysis'].astype('string').str.upper()
    .replace({'CONTROLS': 'CONTROL', 'CONTROL': 'CONTROL', 'ALS': 'ALS'})
)
metadata['cohort_als'] = metadata['cohort'].map({'CONTROL': 0.0, 'ALS': 1.0})
metadata['recording_date'] = pd.to_datetime(
    metadata.get('recording_date_analysis', metadata.get('Recording date')),
    errors='coerce',
)
metadata['age_years'] = safe_numeric(metadata.get('age_at_recording_years'))
metadata['sex'] = metadata.get('Sex', pd.Series(index=metadata.index, dtype='object')).astype('string').str.upper()
metadata['sample_rate_hz'] = safe_numeric(metadata.get('Sampling Rate'))
metadata['protocol_id'] = metadata.get('Protocol ID', pd.Series(index=metadata.index, dtype='object')).astype('string')
metadata['bulbar_score'] = safe_numeric(metadata.get('ALSFRS bulbar subscore'))
metadata['assessment_recording_delta_days'] = safe_numeric(metadata.get('assessment_recording_delta_days'))

if 'assessment_within_primary_window' in metadata.columns:
    window_flag = boolean_series(metadata['assessment_within_primary_window'])
    metadata['bulbar_score_matched'] = metadata['bulbar_score'].where(window_flag)
else:
    metadata['bulbar_score_matched'] = metadata['bulbar_score']

analysis = metadata.merge(
    feature_matrix, on='logical_recording_id', how='inner', validate='one_to_one'
)
analysis = analysis.sort_values(
    ['participant_id', 'recording_date', 'logical_recording_id'], kind='stable'
).reset_index(drop=True)
analysis['recording_sequence'] = analysis.groupby('participant_id').cumcount() + 1
first_date = analysis.groupby('participant_id')['recording_date'].transform('min')
analysis['time_since_first_years'] = (
    (analysis['recording_date'] - first_date).dt.total_seconds()
    / (365.25 * 24 * 60 * 60)
)

print('Analysis matrix:', analysis.shape)


## 5. Blocking population, cohort, date, and bulbar-linkage audit


In [ ]:
audit_rows = []

def add_audit(check: str, observed: Any, expected: Any, passed: bool, detail: str = '', blocking: bool = True):
    audit_rows.append({
        'check': check, 'observed': observed, 'expected': expected,
        'passed': bool(passed), 'blocking': bool(blocking), 'detail': detail,
    })

add_audit('recording count', len(analysis), 519, len(analysis) == 519)
add_audit('participant count', analysis['participant_id'].nunique(), 224,
          analysis['participant_id'].nunique() == 224)
add_audit('unique recording IDs', analysis['logical_recording_id'].nunique(), 519,
          analysis['logical_recording_id'].nunique() == 519)
add_audit('one participant per recording',
          analysis.groupby('logical_recording_id')['participant_id'].nunique().max(), 1,
          analysis.groupby('logical_recording_id')['participant_id'].nunique().max() == 1)
add_audit('cohort constant within participant',
          analysis.groupby('participant_id')['cohort'].nunique().max(), 1,
          analysis.groupby('participant_id')['cohort'].nunique().max() == 1)
add_audit('cohort values', sorted(analysis['cohort'].dropna().unique().tolist()),
          ['ALS', 'CONTROL'], set(analysis['cohort'].dropna()) == {'ALS', 'CONTROL'})
add_audit('missing participant ID', int(analysis['participant_id'].isna().sum()), 0,
          analysis['participant_id'].isna().sum() == 0)
add_audit('missing cohort', int(analysis['cohort'].isna().sum()), 0,
          analysis['cohort'].isna().sum() == 0)
add_audit(
    'missing recording date', int(analysis['recording_date'].isna().sum()),
    'reported; excluded from time-dependent analyses', True,
    detail='Missing dates do not block descriptive/cohort analyses.', blocking=False,
)
if 'freeze_included' in analysis.columns:
    add_audit('all metadata rows freeze included',
              int(analysis['freeze_included'].fillna(False).sum()), 519,
              analysis['freeze_included'].fillna(False).all())

population_audit = pd.DataFrame(audit_rows)
display(population_audit)
export_dataframe(population_audit, LOG_DIR / 'population_audit.csv')
blocking_failures = population_audit.loc[population_audit['blocking'] & ~population_audit['passed']]
if not blocking_failures.empty:
    raise AssertionError('Blocking population audit failed. Inferential analysis is prohibited.')

cohort_summary = (
    analysis.groupby('cohort', observed=True)
    .agg(recordings=('logical_recording_id', 'nunique'),
         participants=('participant_id', 'nunique'))
    .reset_index()
)
recording_count_summary = (
    analysis.groupby(['participant_id', 'cohort'], observed=True)
    .size().rename('recordings').reset_index()
    .groupby('cohort', observed=True)['recordings']
    .agg(['count', 'median', 'min', 'max']).reset_index()
)

als = analysis.loc[analysis['cohort'] == 'ALS'].copy()
bulbar_counts = als.groupby('participant_id')['bulbar_score_matched'].count()
bulbar_distinct = als.groupby('participant_id')['bulbar_score_matched'].nunique()
bulbar_audit = pd.DataFrame({
    'quantity': [
        'ALS recordings', 'ALS recordings with matched bulbar score',
        'ALS participants', 'ALS participants with matched bulbar score',
        'ALS participants with >=2 matched scores',
        'ALS participants with >=2 distinct matched scores',
        'Median absolute assessment-recording difference (days)',
    ],
    'value': [
        len(als), als['bulbar_score_matched'].notna().sum(),
        als['participant_id'].nunique(),
        als.loc[als['bulbar_score_matched'].notna(), 'participant_id'].nunique(),
        bulbar_counts.ge(2).sum(), bulbar_distinct.ge(2).sum(),
        als.loc[als['bulbar_score_matched'].notna(), 'assessment_recording_delta_days'].abs().median(),
    ],
})

display(cohort_summary)
display(recording_count_summary)
display(bulbar_audit)
export_dataframe(cohort_summary, TABLE_DIR / 'population_by_cohort.csv')
export_dataframe(recording_count_summary, TABLE_DIR / 'recordings_per_participant.csv')
export_dataframe(bulbar_audit, TABLE_DIR / 'bulbar_linkage_audit.csv')


## 6. Availability, transforms, scaling, censoring, and zero-state construction


In [ ]:
scaling_rows = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    available_col = f'{feature}__available'
    transformed_col = f'{feature}__transformed'
    z_col = f'{feature}__z'
    raw_z_col = f'{feature}__raw_z'
    positive_col = f'{feature}__positive'

    value_available = analysis[feature].notna()
    if row.availability_field and row.availability_field in analysis.columns:
        explicit_available = boolean_series(analysis[row.availability_field])
        analysis[available_col] = explicit_available & value_available
    else:
        analysis[available_col] = value_available

    transformed = transform_feature(analysis[feature], row.transform).where(analysis[available_col])
    analysis[transformed_col] = transformed
    z, transformed_median, transformed_iqr = robust_scale(transformed)
    analysis[z_col] = z

    raw_values = safe_numeric(analysis[feature]).where(analysis[available_col])
    raw_z, raw_median, raw_iqr = robust_scale(raw_values)
    analysis[raw_z_col] = raw_z
    analysis[positive_col] = analysis[available_col] & safe_numeric(analysis[feature]).gt(0)

    if row.censor_field and row.censor_field in analysis.columns:
        analysis[f'{feature}__censored'] = boolean_series(analysis[row.censor_field]) & analysis[available_col]
    else:
        analysis[f'{feature}__censored'] = False

    scaling_rows.append({
        'feature': feature,
        'family': row.family,
        'transform': row.transform,
        'n_available': int(analysis[available_col].sum()),
        'transformed_median': transformed_median,
        'transformed_iqr': transformed_iqr,
        'raw_median': raw_median,
        'raw_iqr': raw_iqr,
        'degenerate_transformed': not (np.isfinite(transformed_iqr) and transformed_iqr > 0),
        'degenerate_raw': not (np.isfinite(raw_iqr) and raw_iqr > 0),
        'n_censored': int(analysis[f'{feature}__censored'].sum()),
    })

scaling_registry = pd.DataFrame(scaling_rows)
display(scaling_registry)
export_dataframe(scaling_registry, TABLE_DIR / 'scaling_registry.csv')

blocking_degenerate = scaling_registry.merge(
    FEATURE_REGISTRY[['feature', 'outcome_class', 'inference_tier']], on='feature', how='left'
).loc[
    lambda frame: frame['outcome_class'].isin(['continuous', 'censored_continuous', 'one_sided_zero'])
    & frame['inference_tier'].isin(['core', 'secondary'])
    & frame['degenerate_transformed']
]
if not blocking_degenerate.empty:
    raise AssertionError(
        'A core/supporting continuous feature is degenerate after transformation: '
        f"{blocking_degenerate['feature'].tolist()}"
    )


## 7. Recording-level and participant-level descriptive characterization


In [ ]:
def descriptive_summary(data: pd.DataFrame, feature: str, available_col: str) -> dict[str, Any]:
    available = data.loc[data[available_col]].copy()
    values = safe_numeric(available[feature]).dropna()
    if values.empty:
        return {
            'n_recordings': len(data), 'n_participants': data['participant_id'].nunique(),
            'n_available_recordings': 0, 'n_available_participants': 0,
        }
    q = values.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    return {
        'n_recordings': len(data),
        'n_participants': data['participant_id'].nunique(),
        'n_available_recordings': len(available),
        'n_available_participants': available['participant_id'].nunique(),
        'availability_percent': 100 * len(available) / len(data) if len(data) else np.nan,
        'mean': values.mean(), 'standard_deviation': values.std(ddof=1),
        'minimum': values.min(), 'p01': q.loc[0.01], 'p05': q.loc[0.05],
        'q1': q.loc[0.25], 'median': q.loc[0.50], 'q3': q.loc[0.75],
        'p95': q.loc[0.95], 'p99': q.loc[0.99], 'maximum': values.max(),
        'iqr': q.loc[0.75] - q.loc[0.25],
        'bowley_skewness': bowley_skewness(values),
        'zero_count': int(values.eq(0).sum()),
        'zero_percent_available': 100 * values.eq(0).mean(),
        'positive_count': int(values.gt(0).sum()),
        'positive_participants': available.loc[available[feature].gt(0), 'participant_id'].nunique(),
    }

recording_descriptive_rows = []
participant_descriptive_rows = []
positive_extent_rows = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    available_col = f'{feature}__available'
    for cohort in ['ALL', 'CONTROL', 'ALS']:
        subset = analysis if cohort == 'ALL' else analysis.loc[analysis['cohort'] == cohort]
        summary = descriptive_summary(subset, feature, available_col)
        recording_descriptive_rows.append({
            'feature': feature, 'family': row.family, 'label': row.label,
            'validated_role': row.validated_role, 'inference_tier': row.inference_tier,
            'outcome_class': row.outcome_class, 'unit': row.unit,
            'cohort': cohort, **summary,
        })

        participant_medians = (
            subset.loc[subset[available_col]]
            .groupby('participant_id')[feature].median().dropna()
        )
        if participant_medians.empty:
            participant_summary = {}
        else:
            pq = participant_medians.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
            participant_summary = {
                'n_participants_with_value': len(participant_medians),
                'participant_p05': pq.loc[0.05],
                'participant_q1': pq.loc[0.25],
                'participant_median': pq.loc[0.50],
                'participant_q3': pq.loc[0.75],
                'participant_p95': pq.loc[0.95],
                'participant_iqr': pq.loc[0.75] - pq.loc[0.25],
            }
        participant_descriptive_rows.append({
            'feature': feature, 'family': row.family, 'label': row.label,
            'cohort': cohort, **participant_summary,
        })

        positive_subset = subset.loc[
            subset[available_col] & safe_numeric(subset[feature]).gt(0)
        ]
        if not positive_subset.empty:
            positive_values = safe_numeric(positive_subset[feature]).dropna()
            positive_extent_rows.append({
                'feature': feature, 'family': row.family, 'cohort': cohort,
                'n_positive_recordings': len(positive_subset),
                'n_positive_participants': positive_subset['participant_id'].nunique(),
                'positive_median': positive_values.median(),
                'positive_iqr': positive_values.quantile(0.75) - positive_values.quantile(0.25),
                'positive_p95': positive_values.quantile(0.95),
                'positive_maximum': positive_values.max(),
            })

recording_descriptives = pd.DataFrame(recording_descriptive_rows)
participant_descriptives = pd.DataFrame(participant_descriptive_rows)
positive_extent_descriptives = pd.DataFrame(positive_extent_rows)

display(recording_descriptives.head(20))
export_dataframe(recording_descriptives, TABLE_DIR / 'recording_level_descriptives.csv')
export_dataframe(participant_descriptives, TABLE_DIR / 'participant_level_descriptive_sensitivity.csv')
export_dataframe(positive_extent_descriptives, TABLE_DIR / 'positive_extent_descriptives.csv')


## 8. Clustered-model helpers, diagnostics, and prespecified fallbacks


In [ ]:
def _required_columns(rhs: str, data: pd.DataFrame) -> list[str]:
    return [
        token for token in re.findall(r'\b[A-Za-z_]\w*\b', rhs)
        if token in data.columns
    ]


def fit_continuous_clustered(
    data: pd.DataFrame,
    outcome: str,
    rhs: str,
    *,
    feature: str,
    analysis_name: str,
    target_terms: Sequence[str],
    re_formula: str = '1',
    allow_gee_fallback: bool = True,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    required = list(dict.fromkeys([outcome, 'participant_id'] + _required_columns(rhs, data)))
    d = data.dropna(subset=required).copy()
    diagnostics = {
        'feature': feature, 'analysis': analysis_name,
        'n_recordings': len(d), 'n_participants': d['participant_id'].nunique(),
        'attempted_random_effect': re_formula, 'used_model': None,
        'converged': False, 'boundary': False, 'warning_text': '',
    }
    if d.empty or d['participant_id'].nunique() < 2:
        return [], diagnostics | {'warning_text': 'Insufficient data.'}

    formula = f'{outcome} ~ {rhs}'
    mixed_errors = []
    for method in ['lbfgs', 'powell']:
        try:
            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter('always')
                model = smf.mixedlm(
                    formula, data=d, groups=d['participant_id'], re_formula=re_formula
                )
                result = model.fit(reml=False, method=method, maxiter=2000, disp=False)
            warning_text = ' | '.join(str(item.message) for item in captured)
            random_variance = float(result.cov_re.iloc[0, 0]) if result.cov_re.size else np.nan
            covariance_values = np.asarray(result.cov_re, dtype=float)
            eigenvalues = (
                np.linalg.eigvalsh(covariance_values)
                if covariance_values.size else np.asarray([np.nan])
            )
            minimum_eigenvalue = float(np.nanmin(eigenvalues))
            maximum_eigenvalue = float(np.nanmax(eigenvalues))
            condition_number = (
                maximum_eigenvalue / minimum_eigenvalue
                if np.isfinite(minimum_eigenvalue) and minimum_eigenvalue > 0
                else np.inf
            )
            boundary = bool(
                not np.isfinite(minimum_eigenvalue)
                or minimum_eigenvalue < 1e-8
                or condition_number > 1e8
            )
            stable = bool(result.converged and not boundary)
            if stable:
                rows = []
                for target in target_terms:
                    low, high = result.conf_int().loc[target].astype(float)
                    rows.append(model_row(
                        feature=feature, analysis_name=analysis_name, target_term=target,
                        estimate=float(result.params[target]), standard_error=float(result.bse[target]),
                        ci95_low=float(low), ci95_high=float(high),
                        p_value=float(result.pvalues[target]),
                        n_recordings=len(d), n_participants=d['participant_id'].nunique(),
                        model=f'Gaussian mixed model ({method})', converged=True,
                        random_intercept_variance=random_variance,
                        notes=warning_text,
                    ))
                diagnostics.update({
                    'used_model': f'Gaussian mixed model ({method})',
                    'converged': True, 'boundary': boundary,
                    'random_effect_min_eigenvalue': minimum_eigenvalue,
                    'random_effect_condition_number': condition_number,
                    'warning_text': warning_text,
                })
                return rows, diagnostics
            mixed_errors.append(
                f'{method}: converged={result.converged}; random variance={random_variance}; '
                f'min eigenvalue={minimum_eigenvalue}; condition={condition_number}; {warning_text}'
            )
        except Exception as exc:
            mixed_errors.append(f'{method}: {type(exc).__name__}: {exc}')

    if not allow_gee_fallback:
        diagnostics['warning_text'] = ' | '.join(mixed_errors)
        return [], diagnostics

    gee_errors = []
    for covariance in [Exchangeable(), Independence()]:
        try:
            result = sm.GEE.from_formula(
                formula, groups='participant_id', data=d,
                family=sm.families.Gaussian(), cov_struct=covariance,
            ).fit(cov_type='robust')
            rows = []
            for target in target_terms:
                low, high = result.conf_int().loc[target].astype(float)
                rows.append(model_row(
                    feature=feature, analysis_name=analysis_name, target_term=target,
                    estimate=float(result.params[target]), standard_error=float(result.bse[target]),
                    ci95_low=float(low), ci95_high=float(high),
                    p_value=float(result.pvalues[target]),
                    n_recordings=len(d), n_participants=d['participant_id'].nunique(),
                    model=f'Gaussian GEE fallback ({covariance.__class__.__name__})',
                    converged=True, notes='Mixed model fallback: ' + ' | '.join(mixed_errors),
                ))
            diagnostics.update({
                'used_model': f'Gaussian GEE fallback ({covariance.__class__.__name__})',
                'converged': True, 'boundary': True,
                'warning_text': 'Mixed model fallback: ' + ' | '.join(mixed_errors),
            })
            return rows, diagnostics
        except Exception as exc:
            gee_errors.append(f'{covariance.__class__.__name__}: {type(exc).__name__}: {exc}')

    diagnostics['warning_text'] = (
        'Mixed failures: ' + ' | '.join(mixed_errors)
        + '; GEE failures: ' + ' | '.join(gee_errors)
    )
    return [], diagnostics


def fit_binary_gee(
    data: pd.DataFrame,
    outcome: str,
    rhs: str,
    *,
    feature: str,
    analysis_name: str,
    target_terms: Sequence[str],
    enforce_cohort_support: bool = False,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    required = list(dict.fromkeys([outcome, 'participant_id'] + _required_columns(rhs, data)))
    d = data.dropna(subset=required).copy()
    diagnostics = {
        'feature': feature, 'analysis': analysis_name,
        'n_recordings': len(d), 'n_participants': d['participant_id'].nunique(),
        'used_model': None, 'converged': False, 'warning_text': '',
    }
    if d.empty or d['participant_id'].nunique() < 2 or d[outcome].nunique() < 2:
        return [], diagnostics | {'warning_text': 'Insufficient binary variation.'}
    state_support = d.groupby(outcome)['participant_id'].nunique().reindex([0, 1], fill_value=0)
    if state_support.min() < MIN_BINARY_SMALLER_STATE_PARTICIPANTS:
        return [], diagnostics | {'warning_text': f'Binary support gate failed: {state_support.to_dict()}'}
    if enforce_cohort_support:
        support = pd.crosstab(d['cohort'], d[outcome]).reindex(
            index=['CONTROL', 'ALS'], columns=[0, 1], fill_value=0
        )
        if (support == 0).any().any():
            return [], diagnostics | {'warning_text': f'Cohort-by-state support gate failed: {support.to_dict()}'}

    formula = f'{outcome} ~ {rhs}'
    errors = []
    for covariance in [Exchangeable(), Independence()]:
        for covariance_type in ['bias_reduced', 'robust']:
            try:
                result = sm.GEE.from_formula(
                    formula, groups='participant_id', data=d,
                    family=sm.families.Binomial(), cov_struct=covariance,
                ).fit(cov_type=covariance_type)
                rows = []
                for target in target_terms:
                    low, high = result.conf_int().loc[target].astype(float)
                    rows.append(model_row(
                        feature=feature, analysis_name=analysis_name, target_term=target,
                        estimate=float(result.params[target]), standard_error=float(result.bse[target]),
                        ci95_low=float(low), ci95_high=float(high),
                        p_value=float(result.pvalues[target]),
                        n_recordings=len(d), n_participants=d['participant_id'].nunique(),
                        model=(f'Binomial GEE ({covariance.__class__.__name__}; '
                               f'{covariance_type})'), converged=True,
                        notes='Estimate is on the log-odds scale.',
                    ))
                diagnostics.update({
                    'used_model': rows[0]['model'] if rows else None,
                    'converged': True,
                })
                return rows, diagnostics
            except Exception as exc:
                errors.append(
                    f'{covariance.__class__.__name__}/{covariance_type}: '
                    f'{type(exc).__name__}: {exc}'
                )
    diagnostics['warning_text'] = ' | '.join(errors)
    return [], diagnostics


## 9. Measurement availability as a separate outcome (supplementary)


In [ ]:
availability_rows = []
availability_diagnostics = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    available_col = f'{feature}__available'
    availability_rate = float(analysis[available_col].mean())
    if not (0.01 < availability_rate < 0.99):
        continue

    model_data = analysis.copy()
    model_data['availability_outcome'] = model_data[available_col].astype(int)
    rows, diagnostic = fit_binary_gee(
        model_data, 'availability_outcome', 'cohort_als',
        feature=feature, analysis_name='availability_cohort',
        target_terms=['cohort_als'], enforce_cohort_support=True,
    )
    availability_rows.extend(rows)
    availability_diagnostics.append(diagnostic)

    als_data = model_data.loc[
        (model_data['cohort'] == 'ALS')
        & model_data['bulbar_score_matched'].notna()
    ].copy()
    if not als_data.empty:
        participant_mean = als_data.groupby('participant_id')['bulbar_score_matched'].transform('mean')
        als_data['bulbar_between'] = participant_mean - als_data['bulbar_score_matched'].mean()
        als_data['bulbar_within'] = als_data['bulbar_score_matched'] - participant_mean
        varying = als_data.groupby('participant_id')['bulbar_score_matched'].nunique().ge(2).sum()
        rhs = 'bulbar_between + bulbar_within' if varying >= MIN_BULBAR_REPEATED_PARTICIPANTS else 'bulbar_between'
        targets = ['bulbar_between', 'bulbar_within'] if 'bulbar_within' in rhs else ['bulbar_between']
        rows, diagnostic = fit_binary_gee(
            als_data, 'availability_outcome', rhs,
            feature=feature, analysis_name='availability_bulbar', target_terms=targets,
        )
        availability_rows.extend(rows)
        availability_diagnostics.append(diagnostic)

availability_models = pd.DataFrame(availability_rows)
availability_model_diagnostics = pd.DataFrame(availability_diagnostics)
if not availability_models.empty:
    availability_models = availability_models.merge(
        FEATURE_REGISTRY[['feature', 'family', 'inference_tier']], on='feature', how='left'
    )
    availability_models['fdr_family'] = (
        availability_models['analysis'] + '_' + availability_models['target_term']
        + '_' + availability_models['inference_tier']
    )
    availability_models = (
        availability_models.groupby('fdr_family', group_keys=False)
        .apply(lambda frame: bh_adjust(frame), include_groups=False)
        .reset_index(drop=True)
    )

display(availability_models)
export_dataframe(availability_models, TABLE_DIR / 'availability_models.csv')
export_dataframe(availability_model_diagnostics, MODEL_DIR / 'availability_model_diagnostics.csv')


## 10. Participant-level persistence and first-to-second recording agreement


In [ ]:
def fit_icc_point(data: pd.DataFrame, value_col: str) -> tuple[float, float, float, bool, str]:
    d = data.dropna(subset=[value_col, 'participant_id']).copy()
    if d['participant_id'].nunique() < 2:
        return np.nan, np.nan, np.nan, False, 'Insufficient participants.'

    errors: list[str] = []
    for method in ['lbfgs', 'powell', 'bfgs']:
        try:
            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter('always')
                result = smf.mixedlm(
                    f'{value_col} ~ 1', data=d, groups=d['participant_id']
                ).fit(reml=True, method=method, maxiter=3000, disp=False)
            between = float(result.cov_re.iloc[0, 0])
            residual = float(result.scale)
            denominator = between + residual
            icc = between / denominator if denominator > 0 else np.nan
            note = ' | '.join(str(item.message) for item in captured)
            if result.converged and np.isfinite(icc) and between >= 0 and residual > 0:
                return (
                    float(icc), between, residual, True,
                    f'optimizer={method}' + (f' | {note}' if note else ''),
                )
            errors.append(
                f'{method}: converged={result.converged}; between={between}; '
                f'residual={residual}; {note}'
            )
        except Exception as exc:
            errors.append(f'{method}: {type(exc).__name__}: {exc}')

    return np.nan, np.nan, np.nan, False, ' | '.join(errors)


def cluster_bootstrap_icc(
    data: pd.DataFrame,
    value_col: str,
    feature: str,
    n_bootstrap: int,
    seed: int,
) -> np.ndarray:
    checkpoint = BOOTSTRAP_DIR / f'{feature}_icc_bootstrap.csv'
    existing = []
    if checkpoint.exists():
        existing_frame = pd.read_csv(checkpoint)
        existing = existing_frame['icc'].dropna().tolist()
    if len(existing) >= n_bootstrap:
        return np.asarray(existing[:n_bootstrap], dtype=float)

    participants = data['participant_id'].dropna().unique()
    grouped = {pid: frame.copy() for pid, frame in data.groupby('participant_id', sort=False)}
    local_rng = np.random.default_rng(seed)
    estimates = list(existing)
    start = len(estimates)

    for bootstrap_index in range(start, n_bootstrap):
        sampled = local_rng.choice(participants, size=len(participants), replace=True)
        pieces = []
        for duplicate_index, pid in enumerate(sampled):
            piece = grouped[pid].copy()
            piece['participant_id'] = f'{pid}__boot_{bootstrap_index}_{duplicate_index}'
            pieces.append(piece)
        boot = pd.concat(pieces, ignore_index=True)
        icc = fit_icc_point(boot, value_col)[0]
        estimates.append(icc)
        if (bootstrap_index + 1) % 25 == 0 or bootstrap_index + 1 == n_bootstrap:
            pd.DataFrame({'icc': estimates}).to_csv(checkpoint, index=False)
    return np.asarray(estimates, dtype=float)


def first_two_rows(data: pd.DataFrame, value_col: str) -> pd.DataFrame:
    valid = data.dropna(subset=[value_col, 'recording_date']).copy()
    valid = valid.sort_values(['participant_id', 'recording_date', 'logical_recording_id'])
    valid['valid_sequence'] = valid.groupby('participant_id').cumcount() + 1
    wide = valid.loc[valid['valid_sequence'].le(2)].pivot(
        index='participant_id', columns='valid_sequence', values=value_col
    )
    return wide.dropna(subset=[1, 2]) if {1, 2}.issubset(wide.columns) else pd.DataFrame()


def binary_agreement_from_first_two(data: pd.DataFrame, positive_col: str) -> dict[str, Any]:
    pairs = first_two_rows(data, positive_col)
    if pairs.empty:
        return {'n_pairs': 0}
    first = pairs[1].astype(bool)
    second = pairs[2].astype(bool)
    n11 = int((first & second).sum())
    n00 = int((~first & ~second).sum())
    n10 = int((first & ~second).sum())
    n01 = int((~first & second).sum())
    n = len(pairs)
    observed = (n11 + n00) / n
    p_first = first.mean()
    p_second = second.mean()
    expected = p_first * p_second + (1 - p_first) * (1 - p_second)
    kappa = (observed - expected) / (1 - expected) if expected < 1 else np.nan
    positive_agreement = 2 * n11 / (2 * n11 + n10 + n01) if (2 * n11 + n10 + n01) else np.nan
    negative_agreement = 2 * n00 / (2 * n00 + n10 + n01) if (2 * n00 + n10 + n01) else np.nan
    return {
        'n_pairs': n, 'n11': n11, 'n00': n00, 'n10': n10, 'n01': n01,
        'overall_agreement': observed, 'positive_agreement': positive_agreement,
        'negative_agreement': negative_agreement, 'cohen_kappa': kappa,
    }

persistence_rows = []
binary_agreement_rows = []

for row in FEATURE_REGISTRY.loc[FEATURE_REGISTRY['include_persistence']].itertuples(index=False):
    feature = row.feature
    d = analysis.loc[analysis[f'{feature}__available']].copy()
    value_col = f'{feature}__z'
    repeated_participants = d.groupby('participant_id')[value_col].count().ge(2).sum()

    icc, between, residual, converged, note = fit_icc_point(d, value_col)
    if (
        RUN_ICC_BOOTSTRAP
        and repeated_participants >= MIN_REPEATED_PARTICIPANTS
        and np.isfinite(icc)
    ):
        bootstrap = cluster_bootstrap_icc(
            d, value_col, feature, N_ICC_BOOTSTRAP,
            SEED + int(row.feature_order),
        )
        icc_low, icc_high = percentile_interval(bootstrap)
    else:
        icc_low, icc_high = np.nan, np.nan
        if not RUN_ICC_BOOTSTRAP:
            note = (note + ' | ' if note else '') + (
                'ICC bootstrap CI deferred in development mode; set FINAL_RUN=True.'
            )
        elif repeated_participants < MIN_REPEATED_PARTICIPANTS:
            note = (note + ' | ' if note else '') + 'Repeated-support gate failed.'

    pairs = first_two_rows(d, value_col)
    if not pairs.empty:
        rho = float(stats.spearmanr(pairs[1], pairs[2]).statistic)
    else:
        rho = np.nan

    uncensored_icc = np.nan
    uncensored_n = np.nan
    if row.censor_field and row.censor_field in d.columns:
        uncensored = d.loc[~boolean_series(d[row.censor_field])].copy()
        uncensored_icc = fit_icc_point(uncensored, value_col)[0]
        uncensored_n = len(uncensored)

    persistence_rows.append({
        'feature': feature, 'family': row.family, 'label': row.label,
        'validated_role': row.validated_role, 'inference_tier': row.inference_tier,
        'n_recordings': len(d), 'n_participants': d['participant_id'].nunique(),
        'n_repeated_participants': int(repeated_participants),
        'icc': icc, 'icc_ci95_low': icc_low, 'icc_ci95_high': icc_high,
        'between_participant_variance': between, 'residual_variance': residual,
        'icc_model_converged': converged, 'icc_note': note,
        'uncensored_icc_sensitivity': uncensored_icc,
        'uncensored_n_recordings': uncensored_n,
        'first_second_spearman': rho, 'n_first_second_pairs': len(pairs),
    })

    if row.outcome_class == 'one_sided_zero':
        agreement = binary_agreement_from_first_two(d, f'{feature}__positive')
        binary_agreement_rows.append({
            'feature': feature, 'family': row.family, **agreement,
        })

# Sparse and monitoring families receive agreement summaries but no ICC.
for row in FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['outcome_class'].isin(['sparse_event_extent', 'monitoring'])
    & FEATURE_REGISTRY['include_main']
].itertuples(index=False):
    d = analysis.loc[analysis[f'{row.feature}__available']].copy()
    agreement = binary_agreement_from_first_two(d, f'{row.feature}__positive')
    binary_agreement_rows.append({
        'feature': row.feature, 'family': row.family, **agreement,
    })

persistence_table = pd.DataFrame(persistence_rows)
binary_agreement_table = pd.DataFrame(binary_agreement_rows)
display(persistence_table)
display(binary_agreement_table)
export_dataframe(persistence_table, TABLE_DIR / 'participant_level_persistence.csv')
export_dataframe(binary_agreement_table, TABLE_DIR / 'binary_first_second_agreement.csv')


## 11. ALS-only longitudinal change


In [ ]:
longitudinal_rows = []
longitudinal_diagnostics = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    d = analysis.loc[
        (analysis['cohort'] == 'ALS')
        & analysis[f'{feature}__available']
        & analysis['time_since_first_years'].notna()
    ].copy()
    if d.empty:
        continue

    repeated = d.groupby('participant_id').size().ge(2).sum()
    three_plus = d.groupby('participant_id').size().ge(3).sum()
    if repeated < MIN_REPEATED_PARTICIPANTS or len(d) < MIN_REPEATED_JOINT_OBSERVATIONS:
        continue

    if row.outcome_class in ['continuous', 'censored_continuous']:
        re_formula = '~time_since_first_years' if (
            repeated >= MIN_RANDOM_SLOPE_PARTICIPANTS
            and three_plus >= MIN_RANDOM_SLOPE_THREE_RECORDINGS
        ) else '1'
        # Attempt the random slope only when its support gate passes. A failed or
        # singular random-slope fit is followed by the prespecified random-intercept
        # model before any marginal GEE fallback is allowed.
        if re_formula != '1':
            rows, diagnostic = fit_continuous_clustered(
                d, f'{feature}__z', 'time_since_first_years',
                feature=feature, analysis_name='longitudinal_continuous_random_slope',
                target_terms=['time_since_first_years'], re_formula=re_formula,
                allow_gee_fallback=False,
            )
            longitudinal_diagnostics.append(diagnostic)
        else:
            rows = []

        if not rows:
            rows, diagnostic = fit_continuous_clustered(
                d, f'{feature}__z', 'time_since_first_years',
                feature=feature, analysis_name='longitudinal_continuous',
                target_terms=['time_since_first_years'], re_formula='1',
            )
            longitudinal_diagnostics.append(diagnostic)
        else:
            for fitted_row in rows:
                fitted_row['analysis'] = 'longitudinal_continuous'
                fitted_row['notes'] = (
                    (fitted_row.get('notes', '') + ' | ') if fitted_row.get('notes') else ''
                ) + 'Random-slope model retained.'

        longitudinal_rows.extend(rows)

        if row.censor_field and row.censor_field in d.columns:
            uncensored = d.loc[~boolean_series(d[row.censor_field])].copy()
            sensitivity_rows, sensitivity_diagnostic = fit_continuous_clustered(
                uncensored, f'{feature}__z', 'time_since_first_years',
                feature=feature,
                analysis_name='longitudinal_continuous_uncensored_sensitivity',
                target_terms=['time_since_first_years'], re_formula='1',
            )
            longitudinal_rows.extend(sensitivity_rows)
            longitudinal_diagnostics.append(sensitivity_diagnostic)

    elif row.outcome_class == 'one_sided_zero':
        d['positive_presence'] = d[f'{feature}__positive'].astype(int)
        rows, diagnostic = fit_binary_gee(
            d, 'positive_presence', 'time_since_first_years',
            feature=feature, analysis_name='longitudinal_presence',
            target_terms=['time_since_first_years'],
        )
        longitudinal_rows.extend(rows)
        longitudinal_diagnostics.append(diagnostic)

        positive = d.loc[d['positive_presence'] == 1].copy()
        if (
            len(positive) >= MIN_REPEATED_JOINT_OBSERVATIONS
            and positive['participant_id'].nunique() >= MIN_SPARSE_POSITIVE_PARTICIPANTS
        ):
            rows, diagnostic = fit_continuous_clustered(
                positive, f'{feature}__z', 'time_since_first_years',
                feature=feature, analysis_name='longitudinal_positive_extent',
                target_terms=['time_since_first_years'],
            )
            longitudinal_rows.extend(rows)
            longitudinal_diagnostics.append(diagnostic)

longitudinal_models = pd.DataFrame(longitudinal_rows)
longitudinal_model_diagnostics = pd.DataFrame(longitudinal_diagnostics)
if not longitudinal_models.empty:
    longitudinal_models = longitudinal_models.merge(
        FEATURE_REGISTRY[['feature', 'family', 'inference_tier']], on='feature', how='left'
    )
    longitudinal_models['fdr_family'] = (
        longitudinal_models['analysis'] + '_' + longitudinal_models['inference_tier']
    )
    longitudinal_models = (
        longitudinal_models.groupby('fdr_family', group_keys=False)
        .apply(lambda frame: bh_adjust(frame), include_groups=False)
        .reset_index(drop=True)
    )

display(longitudinal_models)
export_dataframe(longitudinal_models, TABLE_DIR / 'als_longitudinal_models.csv')
export_dataframe(longitudinal_model_diagnostics, MODEL_DIR / 'als_longitudinal_model_diagnostics.csv')


## 12. ALS-control cohort differences, routed by outcome class


In [ ]:
cohort_rows = []
cohort_diagnostics = []
sparse_gate_rows = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    d = analysis.loc[analysis[f'{feature}__available']].copy()
    if d.empty:
        continue

    if row.outcome_class in ['continuous', 'censored_continuous']:
        rows, diagnostic = fit_continuous_clustered(
            d, f'{feature}__z', 'cohort_als',
            feature=feature, analysis_name='cohort_continuous',
            target_terms=['cohort_als'],
        )
        cohort_rows.extend(rows)
        cohort_diagnostics.append(diagnostic)

        if row.censor_field and row.censor_field in d.columns:
            uncensored = d.loc[~boolean_series(d[row.censor_field])].copy()
            rows, diagnostic = fit_continuous_clustered(
                uncensored, f'{feature}__z', 'cohort_als',
                feature=feature, analysis_name='cohort_continuous_uncensored_sensitivity',
                target_terms=['cohort_als'],
            )
            cohort_rows.extend(rows)
            cohort_diagnostics.append(diagnostic)

    elif row.outcome_class == 'one_sided_zero':
        d['positive_presence'] = d[f'{feature}__positive'].astype(int)
        rows, diagnostic = fit_binary_gee(
            d, 'positive_presence', 'cohort_als',
            feature=feature, analysis_name='cohort_presence',
            target_terms=['cohort_als'], enforce_cohort_support=True,
        )
        cohort_rows.extend(rows)
        cohort_diagnostics.append(diagnostic)

        positive = d.loc[d['positive_presence'] == 1].copy()
        positive_by_cohort = positive.groupby('cohort')['participant_id'].nunique().reindex(
            ['CONTROL', 'ALS'], fill_value=0
        )
        if (
            len(positive) >= MIN_SPARSE_POSITIVE_RECORDINGS
            and positive['participant_id'].nunique() >= MIN_SPARSE_POSITIVE_PARTICIPANTS
            and positive_by_cohort.min() >= MIN_BINARY_SMALLER_STATE_PARTICIPANTS
        ):
            rows, diagnostic = fit_continuous_clustered(
                positive, f'{feature}__z', 'cohort_als',
                feature=feature, analysis_name='cohort_positive_extent',
                target_terms=['cohort_als'],
            )
            cohort_rows.extend(rows)
            cohort_diagnostics.append(diagnostic)

    elif row.outcome_class in ['sparse_event_extent', 'sparse_event_rate', 'monitoring']:
        positive = d[f'{feature}__positive']
        gate = {
            'feature': feature, 'family': row.family,
            'n_available_recordings': len(d),
            'n_positive_recordings': int(positive.sum()),
            'n_positive_participants': d.loc[positive, 'participant_id'].nunique(),
            'positive_cohorts': d.loc[positive, 'cohort'].nunique(),
        }
        gate['passes_inferential_gate'] = bool(
            gate['n_positive_recordings'] >= MIN_SPARSE_POSITIVE_RECORDINGS
            and gate['n_positive_participants'] >= MIN_SPARSE_POSITIVE_PARTICIPANTS
            and gate['positive_cohorts'] == 2
        )
        sparse_gate_rows.append(gate)

cohort_models = pd.DataFrame(cohort_rows)
cohort_model_diagnostics = pd.DataFrame(cohort_diagnostics)
sparse_model_gates = pd.DataFrame(sparse_gate_rows)
if not cohort_models.empty:
    cohort_models = cohort_models.merge(
        FEATURE_REGISTRY[['feature', 'family', 'inference_tier']], on='feature', how='left'
    )
    cohort_models['fdr_family'] = (
        cohort_models['analysis'] + '_' + cohort_models['inference_tier']
    )
    cohort_models = (
        cohort_models.groupby('fdr_family', group_keys=False)
        .apply(lambda frame: bh_adjust(frame), include_groups=False)
        .reset_index(drop=True)
    )

display(cohort_models)
display(sparse_model_gates)
export_dataframe(cohort_models, TABLE_DIR / 'cohort_models.csv')
export_dataframe(cohort_model_diagnostics, MODEL_DIR / 'cohort_model_diagnostics.csv')
export_dataframe(sparse_model_gates, TABLE_DIR / 'sparse_event_model_gates.csv')


## 13. Demographic, sample-rate, untransformed, and winsorized cohort sensitivities


In [ ]:
sensitivity_rows = []
sensitivity_diagnostics = []

# ------------------------------------------------------------------
# Continuous and censored-continuous indicators
# ------------------------------------------------------------------
for row in FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['outcome_class'].isin(['continuous', 'censored_continuous'])
].itertuples(index=False):
    feature = row.feature
    d = analysis.loc[analysis[f'{feature}__available']].copy()

    d['age_centered'] = d['age_years'] - d['age_years'].mean(skipna=True)
    d['sex_female'] = d['sex'].map({'MALE': 0.0, 'FEMALE': 1.0})
    d['sample_rate_48k'] = d['sample_rate_hz'].eq(48000).astype(float)

    # Demographic adjustment is interpreted only beside an unadjusted
    # model fitted to the identical complete-case subset. This separates
    # covariate adjustment from changes caused by sample restriction.
    demographic_complete = d.dropna(
        subset=[
            f'{feature}__z',
            'participant_id',
            'cohort_als',
            'age_centered',
            'sex_female',
        ]
    ).copy()

    if (
        len(demographic_complete) >= 50
        and demographic_complete['participant_id'].nunique() >= 2
    ):
        rows, diagnostic = fit_continuous_clustered(
            demographic_complete,
            f'{feature}__z',
            'cohort_als',
            feature=feature,
            analysis_name='cohort_demographic_complete_case_unadjusted',
            target_terms=['cohort_als'],
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)

        rows, diagnostic = fit_continuous_clustered(
            demographic_complete,
            f'{feature}__z',
            'cohort_als + age_centered + sex_female',
            feature=feature,
            analysis_name='cohort_demographic_adjusted',
            target_terms=['cohort_als'],
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)

    if d['sample_rate_hz'].nunique(dropna=True) >= 2:
        rows, diagnostic = fit_continuous_clustered(
            d,
            f'{feature}__z',
            'cohort_als + sample_rate_48k',
            feature=feature,
            analysis_name='cohort_sample_rate_adjusted',
            target_terms=['cohort_als'],
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)

    rows, diagnostic = fit_continuous_clustered(
        d,
        f'{feature}__raw_z',
        'cohort_als',
        feature=feature,
        analysis_name='cohort_untransformed_sensitivity',
        target_terms=['cohort_als'],
    )
    sensitivity_rows.extend(rows)
    sensitivity_diagnostics.append(diagnostic)

    winsor_col = f'{feature}__winsor_z'
    low, high = d[f'{feature}__z'].quantile(WINSOR_LIMITS)
    d[winsor_col] = d[f'{feature}__z'].clip(
        lower=low,
        upper=high,
    )

    rows, diagnostic = fit_continuous_clustered(
        d,
        winsor_col,
        'cohort_als',
        feature=feature,
        analysis_name='cohort_winsorized_sensitivity',
        target_terms=['cohort_als'],
    )
    sensitivity_rows.extend(rows)
    sensitivity_diagnostics.append(diagnostic)


# ------------------------------------------------------------------
# One-sided QCHAN state indicators
# ------------------------------------------------------------------
for row in FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['outcome_class'].eq('one_sided_zero')
].itertuples(index=False):
    feature = row.feature
    d = analysis.loc[analysis[f'{feature}__available']].copy()

    d['positive_presence'] = d[f'{feature}__positive'].astype(int)
    d['age_centered'] = d['age_years'] - d['age_years'].mean(skipna=True)
    d['sex_female'] = d['sex'].map({'MALE': 0.0, 'FEMALE': 1.0})
    d['sample_rate_48k'] = d['sample_rate_hz'].eq(48000).astype(float)

    demographic_complete = d.dropna(
        subset=[
            'positive_presence',
            'participant_id',
            'cohort_als',
            'age_centered',
            'sex_female',
        ]
    ).copy()

    if (
        len(demographic_complete) >= 50
        and demographic_complete['participant_id'].nunique() >= 2
    ):
        rows, diagnostic = fit_binary_gee(
            demographic_complete,
            'positive_presence',
            'cohort_als',
            feature=feature,
            analysis_name=(
                'cohort_demographic_complete_case_unadjusted_presence'
            ),
            target_terms=['cohort_als'],
            enforce_cohort_support=True,
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)

        rows, diagnostic = fit_binary_gee(
            demographic_complete,
            'positive_presence',
            'cohort_als + age_centered + sex_female',
            feature=feature,
            analysis_name='cohort_demographic_adjusted_presence',
            target_terms=['cohort_als'],
            enforce_cohort_support=True,
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)

    if d['sample_rate_hz'].nunique(dropna=True) >= 2:
        rows, diagnostic = fit_binary_gee(
            d,
            'positive_presence',
            'cohort_als + sample_rate_48k',
            feature=feature,
            analysis_name='cohort_sample_rate_adjusted_presence',
            target_terms=['cohort_als'],
            enforce_cohort_support=True,
        )
        sensitivity_rows.extend(rows)
        sensitivity_diagnostics.append(diagnostic)


cohort_sensitivity_models = pd.DataFrame(
    sensitivity_rows
)
cohort_sensitivity_diagnostics = pd.DataFrame(
    sensitivity_diagnostics
)

display(cohort_sensitivity_models)

export_dataframe(
    cohort_sensitivity_models,
    TABLE_DIR / 'cohort_sensitivity_models.csv',
)
export_dataframe(
    cohort_sensitivity_diagnostics,
    MODEL_DIR / 'cohort_sensitivity_diagnostics.csv',
)


# ------------------------------------------------------------------
# Direct demographic-adjustment audit on identical complete cases
# ------------------------------------------------------------------
demographic_pairs = []

for feature, group in cohort_sensitivity_models.groupby(
    'feature',
    sort=False,
):
    analysis_names = set(group['analysis'])

    if {
        'cohort_demographic_complete_case_unadjusted',
        'cohort_demographic_adjusted',
    }.issubset(analysis_names):
        unadjusted = group.loc[
            group['analysis'].eq(
                'cohort_demographic_complete_case_unadjusted'
            )
        ].iloc[0]

        adjusted = group.loc[
            group['analysis'].eq(
                'cohort_demographic_adjusted'
            )
        ].iloc[0]

    elif {
        'cohort_demographic_complete_case_unadjusted_presence',
        'cohort_demographic_adjusted_presence',
    }.issubset(analysis_names):
        unadjusted = group.loc[
            group['analysis'].eq(
                'cohort_demographic_complete_case_unadjusted_presence'
            )
        ].iloc[0]

        adjusted = group.loc[
            group['analysis'].eq(
                'cohort_demographic_adjusted_presence'
            )
        ].iloc[0]

    else:
        continue

    if (
        int(unadjusted['n_recordings'])
        != int(adjusted['n_recordings'])
        or int(unadjusted['n_participants'])
        != int(adjusted['n_participants'])
    ):
        raise AssertionError(
            f'Demographic models use different samples for {feature}.'
        )

    unadjusted_estimate = float(
        unadjusted['estimate']
    )
    adjusted_estimate = float(
        adjusted['estimate']
    )

    demographic_pairs.append(
        {
            'feature': feature,
            'unadjusted_analysis': unadjusted['analysis'],
            'adjusted_analysis': adjusted['analysis'],
            'n_recordings': int(
                adjusted['n_recordings']
            ),
            'n_participants': int(
                adjusted['n_participants']
            ),
            'unadjusted_estimate': unadjusted_estimate,
            'adjusted_estimate': adjusted_estimate,
            'adjusted_minus_unadjusted': (
                adjusted_estimate
                - unadjusted_estimate
            ),
            'same_direction': bool(
                np.sign(unadjusted_estimate)
                == np.sign(adjusted_estimate)
            ),
            'unadjusted_ci95_low': float(
                unadjusted['ci95_low']
            ),
            'unadjusted_ci95_high': float(
                unadjusted['ci95_high']
            ),
            'adjusted_ci95_low': float(
                adjusted['ci95_low']
            ),
            'adjusted_ci95_high': float(
                adjusted['ci95_high']
            ),
            'unadjusted_p_value': float(
                unadjusted['p_value']
            ),
            'adjusted_p_value': float(
                adjusted['p_value']
            ),
        }
    )

demographic_adjustment_comparison = pd.DataFrame(
    demographic_pairs
)

display(demographic_adjustment_comparison)

export_dataframe(
    demographic_adjustment_comparison,
    TABLE_DIR / 'cohort_demographic_adjustment_comparison.csv',
)

## 14. Recording-matched bulbar effects: between participant, within participant, and time-adjusted sensitivity


In [ ]:
bulbar_rows = []
bulbar_diagnostics = []
bulbar_gate_rows = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    d = analysis.loc[
        (analysis['cohort'] == 'ALS')
        & analysis[f'{feature}__available']
        & analysis['bulbar_score_matched'].notna()
    ].copy()
    if d.empty:
        continue

    participant_mean = d.groupby('participant_id')['bulbar_score_matched'].transform('mean')
    grand_mean = float(d['bulbar_score_matched'].mean())
    d['bulbar_between'] = participant_mean - grand_mean
    d['bulbar_within'] = d['bulbar_score_matched'] - participant_mean

    counts = d.groupby('participant_id')['bulbar_score_matched'].count()
    distinct = d.groupby('participant_id')['bulbar_score_matched'].nunique()
    between_gate = bool(
        len(d) >= MIN_BULBAR_MATCHED_RECORDINGS
        and d['participant_id'].nunique() >= 30
    )
    within_gate = bool(
        between_gate
        and counts.ge(2).sum() >= MIN_BULBAR_REPEATED_PARTICIPANTS
        and distinct.ge(2).sum() >= MIN_BULBAR_REPEATED_PARTICIPANTS
    )
    bulbar_gate_rows.append({
        'feature': feature, 'family': row.family,
        'n_matched_recordings': len(d),
        'n_matched_participants': d['participant_id'].nunique(),
        'n_repeated_participants': counts.ge(2).sum(),
        'n_within_varying_participants': distinct.ge(2).sum(),
        'between_gate': between_gate, 'within_gate': within_gate,
    })
    if not between_gate:
        continue

    rhs = 'bulbar_between + bulbar_within' if within_gate else 'bulbar_between'
    targets = ['bulbar_between', 'bulbar_within'] if within_gate else ['bulbar_between']

    if row.outcome_class in ['continuous', 'censored_continuous']:
        rows, diagnostic = fit_continuous_clustered(
            d, f'{feature}__z', rhs,
            feature=feature, analysis_name='bulbar_continuous',
            target_terms=targets,
        )
        bulbar_rows.extend(rows)
        bulbar_diagnostics.append(diagnostic)

        if within_gate:
            rows, diagnostic = fit_continuous_clustered(
                d, f'{feature}__z', rhs + ' + time_since_first_years',
                feature=feature, analysis_name='bulbar_continuous_time_adjusted',
                target_terms=targets,
            )
            bulbar_rows.extend(rows)
            bulbar_diagnostics.append(diagnostic)

        if row.censor_field and row.censor_field in d.columns:
            uncensored = d.loc[~boolean_series(d[row.censor_field])].copy()
            rows, diagnostic = fit_continuous_clustered(
                uncensored, f'{feature}__z', rhs,
                feature=feature, analysis_name='bulbar_continuous_uncensored_sensitivity',
                target_terms=targets,
            )
            bulbar_rows.extend(rows)
            bulbar_diagnostics.append(diagnostic)

    elif row.outcome_class == 'one_sided_zero':
        d['positive_presence'] = d[f'{feature}__positive'].astype(int)
        rows, diagnostic = fit_binary_gee(
            d, 'positive_presence', rhs,
            feature=feature, analysis_name='bulbar_presence', target_terms=targets,
        )
        bulbar_rows.extend(rows)
        bulbar_diagnostics.append(diagnostic)

        if within_gate:
            rows, diagnostic = fit_binary_gee(
                d, 'positive_presence', rhs + ' + time_since_first_years',
                feature=feature, analysis_name='bulbar_presence_time_adjusted',
                target_terms=targets,
            )
            bulbar_rows.extend(rows)
            bulbar_diagnostics.append(diagnostic)

        positive = d.loc[d['positive_presence'] == 1].copy()
        if (
            len(positive) >= MIN_SPARSE_POSITIVE_RECORDINGS
            and positive['participant_id'].nunique() >= MIN_SPARSE_POSITIVE_PARTICIPANTS
        ):
            positive_participant_mean = positive.groupby('participant_id')['bulbar_score_matched'].transform('mean')
            positive['bulbar_between'] = positive_participant_mean - positive['bulbar_score_matched'].mean()
            positive['bulbar_within'] = positive['bulbar_score_matched'] - positive_participant_mean
            positive_distinct = positive.groupby('participant_id')['bulbar_score_matched'].nunique().ge(2).sum()
            positive_rhs = (
                'bulbar_between + bulbar_within'
                if positive_distinct >= MIN_BULBAR_REPEATED_PARTICIPANTS
                else 'bulbar_between'
            )
            positive_targets = (
                ['bulbar_between', 'bulbar_within']
                if 'bulbar_within' in positive_rhs else ['bulbar_between']
            )
            rows, diagnostic = fit_continuous_clustered(
                positive, f'{feature}__z', positive_rhs,
                feature=feature, analysis_name='bulbar_positive_extent',
                target_terms=positive_targets,
            )
            bulbar_rows.extend(rows)
            bulbar_diagnostics.append(diagnostic)

bulbar_models = pd.DataFrame(bulbar_rows)
bulbar_model_diagnostics = pd.DataFrame(bulbar_diagnostics)
bulbar_model_gates = pd.DataFrame(bulbar_gate_rows)
if not bulbar_models.empty:
    bulbar_models = bulbar_models.merge(
        FEATURE_REGISTRY[['feature', 'family', 'inference_tier']], on='feature', how='left'
    )
    bulbar_models['fdr_family'] = (
        bulbar_models['analysis'] + '_' + bulbar_models['target_term']
        + '_' + bulbar_models['inference_tier']
    )
    bulbar_models = (
        bulbar_models.groupby('fdr_family', group_keys=False)
        .apply(lambda frame: bh_adjust(frame), include_groups=False)
        .reset_index(drop=True)
    )

display(bulbar_model_gates)
display(bulbar_models)
export_dataframe(bulbar_model_gates, TABLE_DIR / 'bulbar_model_gates.csv')
export_dataframe(bulbar_models, TABLE_DIR / 'bulbar_models.csv')
export_dataframe(bulbar_model_diagnostics, MODEL_DIR / 'bulbar_model_diagnostics.csv')


## 15. Participant-collapsed and one-recording-per-participant cohort sensitivities


In [ ]:
participant_collapsed_rows = []
resampling_rows = []

for row in FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['outcome_class'].isin(['continuous', 'censored_continuous', 'one_sided_zero'])
].itertuples(index=False):
    feature = row.feature
    d = analysis.loc[analysis[f'{feature}__available']].copy()

    if row.outcome_class in ['continuous', 'censored_continuous']:
        collapsed = (
            d.groupby(['participant_id', 'cohort'], observed=True)[f'{feature}__z']
            .median().reset_index()
        )
        if collapsed['cohort'].nunique() == 2:
            result = smf.ols(f'Q("{feature}__z") ~ C(cohort)', data=collapsed).fit(cov_type='HC3')
            target = 'C(cohort)[T.CONTROL]' if 'C(cohort)[T.CONTROL]' in result.params else 'C(cohort)[T.ALS]'
            estimate = float(result.params[target])
            if target.endswith('CONTROL]'):
                estimate *= -1
            participant_collapsed_rows.append({
                'feature': feature, 'family': row.family, 'analysis': 'participant_collapsed_continuous',
                'estimate_als_minus_control': estimate, 'p_value': float(result.pvalues[target]),
                'n_participants': len(collapsed),
            })

        estimates = []
        local_rng = np.random.default_rng(SEED + 10000 + int(row.feature_order))
        participant_indices = [
            group.index.to_numpy()
            for _, group in d.groupby('participant_id', sort=False)
        ]
        for iteration in range(N_ONE_RECORDING_RESAMPLES):
            selected_indices = np.fromiter(
                (local_rng.choice(indices) for indices in participant_indices),
                dtype=int,
                count=len(participant_indices),
            )
            selected = d.loc[selected_indices]
            if selected['cohort'].nunique() < 2:
                continue
            fit = smf.ols(f'Q("{feature}__z") ~ cohort_als', data=selected).fit(cov_type='HC3')
            estimates.append(float(fit.params['cohort_als']))
        low, high = percentile_interval(estimates)
        resampling_rows.append({
            'feature': feature, 'family': row.family,
            'analysis': 'one_recording_per_participant_continuous',
            'median_estimate': np.nanmedian(estimates),
            'ci95_low': low, 'ci95_high': high,
            'n_resamples': len(estimates),
        })

    elif row.outcome_class == 'one_sided_zero':
        # Equal-participant sensitivity: each participant contributes the observed
        # proportion of positive recordings rather than an ever-positive flag.
        collapsed = (
            d.groupby(['participant_id', 'cohort', 'cohort_als'], observed=True)[f'{feature}__positive']
            .mean().reset_index(name='participant_positive_proportion')
        )
        if collapsed['cohort'].nunique() == 2:
            result = smf.ols(
                'participant_positive_proportion ~ cohort_als', data=collapsed
            ).fit(cov_type='HC3')
            participant_collapsed_rows.append({
                'feature': feature, 'family': row.family,
                'analysis': 'participant_collapsed_presence_proportion',
                'estimate_als_minus_control': float(result.params['cohort_als']),
                'p_value': float(result.pvalues['cohort_als']),
                'n_participants': len(collapsed),
                'notes': 'Difference in participant-level positive-recording proportion.',
            })

        estimates = []
        local_rng = np.random.default_rng(SEED + 20000 + int(row.feature_order))
        participant_indices = [
            group.index.to_numpy()
            for _, group in d.groupby('participant_id', sort=False)
        ]
        for iteration in range(N_ONE_RECORDING_RESAMPLES):
            selected_indices = np.fromiter(
                (local_rng.choice(indices) for indices in participant_indices),
                dtype=int, count=len(participant_indices),
            )
            selected = d.loc[selected_indices].copy()
            selected['selected_presence'] = selected[f'{feature}__positive'].astype(int)
            support = pd.crosstab(selected['cohort'], selected['selected_presence']).reindex(
                index=['CONTROL', 'ALS'], columns=[0, 1], fill_value=0
            )
            if (support == 0).any().any():
                continue
            try:
                fit = smf.logit('selected_presence ~ cohort_als', data=selected).fit(disp=False)
                estimates.append(float(fit.params['cohort_als']))
            except Exception:
                continue
        low, high = percentile_interval(estimates)
        resampling_rows.append({
            'feature': feature, 'family': row.family,
            'analysis': 'one_recording_per_participant_presence_log_odds',
            'median_estimate': np.nanmedian(estimates) if estimates else np.nan,
            'ci95_low': low, 'ci95_high': high,
            'n_resamples': len(estimates),
        })

participant_collapsed_sensitivity = pd.DataFrame(participant_collapsed_rows)
one_recording_resampling = pd.DataFrame(resampling_rows)
if not participant_collapsed_sensitivity.empty:
    participant_collapsed_sensitivity = participant_collapsed_sensitivity.merge(
        FEATURE_REGISTRY[['feature', 'inference_tier']], on='feature', how='left'
    )
    participant_collapsed_sensitivity['fdr_family'] = (
        participant_collapsed_sensitivity['analysis'] + '_'
        + participant_collapsed_sensitivity['inference_tier']
    )
    participant_collapsed_sensitivity = (
        participant_collapsed_sensitivity.groupby('fdr_family', group_keys=False)
        .apply(lambda frame: bh_adjust(frame), include_groups=False)
        .reset_index(drop=True)
    )

display(participant_collapsed_sensitivity)
display(one_recording_resampling)
export_dataframe(participant_collapsed_sensitivity, TABLE_DIR / 'participant_collapsed_cohort_sensitivity.csv')
export_dataframe(one_recording_resampling, TABLE_DIR / 'one_recording_per_participant_resampling.csv')


## 16. Main figure source data


In [ ]:
MAIN_CONTINUOUS = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['include_main']
    & FEATURE_REGISTRY['outcome_class'].isin(['continuous', 'censored_continuous'])
    & FEATURE_REGISTRY['inference_tier'].isin(['core', 'secondary'])
].sort_values(['family', 'feature_order'])

MAIN_ONE_SIDED = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['include_main']
    & FEATURE_REGISTRY['outcome_class'].eq('one_sided_zero')
    & FEATURE_REGISTRY['inference_tier'].isin(['core', 'secondary'])
].sort_values(['family', 'feature_order'])

MAIN_SPARSE = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['include_main']
    & FEATURE_REGISTRY['outcome_class'].isin(['sparse_event_extent', 'monitoring'])
].sort_values(['family', 'feature_order'])

panel_a_continuous = recording_descriptives.loc[
    recording_descriptives['feature'].isin(MAIN_CONTINUOUS['feature'])
    & recording_descriptives['cohort'].isin(['CONTROL', 'ALS'])
].merge(
    MAIN_CONTINUOUS[['feature', 'short_label', 'feature_order', 'inference_tier', 'direction']],
    on='feature', how='left'
)

panel_a_states = recording_descriptives.loc[
    recording_descriptives['feature'].isin(MAIN_ONE_SIDED['feature'])
    & recording_descriptives['cohort'].isin(['CONTROL', 'ALS'])
].merge(
    MAIN_ONE_SIDED[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_a_sparse = recording_descriptives.loc[
    recording_descriptives['feature'].isin(MAIN_SPARSE['feature'])
    & recording_descriptives['cohort'].isin(['CONTROL', 'ALS'])
].merge(
    MAIN_SPARSE[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_b_persistence = persistence_table.loc[
    persistence_table['feature'].isin(
        pd.concat([MAIN_CONTINUOUS, MAIN_ONE_SIDED])['feature']
    )
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_b_longitudinal_continuous = longitudinal_models.loc[
    longitudinal_models['analysis'].eq('longitudinal_continuous')
    & longitudinal_models['feature'].isin(MAIN_CONTINUOUS['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_b_longitudinal_presence = longitudinal_models.loc[
    longitudinal_models['analysis'].eq('longitudinal_presence')
    & longitudinal_models['feature'].isin(MAIN_ONE_SIDED['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_c_cohort_continuous = cohort_models.loc[
    cohort_models['analysis'].eq('cohort_continuous')
    & cohort_models['feature'].isin(MAIN_CONTINUOUS['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_c_bulbar_continuous = bulbar_models.loc[
    bulbar_models['analysis'].eq('bulbar_continuous')
    & bulbar_models['feature'].isin(MAIN_CONTINUOUS['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_c_cohort_presence = cohort_models.loc[
    cohort_models['analysis'].eq('cohort_presence')
    & cohort_models['feature'].isin(MAIN_ONE_SIDED['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_c_bulbar_presence = bulbar_models.loc[
    bulbar_models['analysis'].eq('bulbar_presence')
    & bulbar_models['feature'].isin(MAIN_ONE_SIDED['feature'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)

panel_a_standardized_rows = []
for row in MAIN_CONTINUOUS.itertuples(index=False):
    feature = row.feature
    for cohort in ['CONTROL', 'ALS']:
        mask = (
            (analysis['cohort'] == cohort)
            & analysis[f'{feature}__available']
            & analysis[f'{feature}__z'].notna()
        )
        values = analysis.loc[mask, f'{feature}__z']
        if values.empty:
            continue
        quantiles = values.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
        panel_a_standardized_rows.append({
            'feature': feature, 'family': row.family,
            'short_label': row.short_label, 'feature_order': row.feature_order,
            'inference_tier': row.inference_tier, 'cohort': cohort,
            'n_recordings': len(values),
            'n_participants': analysis.loc[mask, 'participant_id'].nunique(),
            'p05_z': quantiles.loc[0.05], 'q1_z': quantiles.loc[0.25],
            'median_z': quantiles.loc[0.50], 'q3_z': quantiles.loc[0.75],
            'p95_z': quantiles.loc[0.95],
        })
panel_a_standardized = pd.DataFrame(panel_a_standardized_rows)

for name, frame in {
    'panel_A_continuous_raw_units': panel_a_continuous,
    'panel_A_continuous_standardized': panel_a_standardized,
    'panel_A_states': panel_a_states,
    'panel_A_sparse': panel_a_sparse,
    'panel_B_persistence': panel_b_persistence,
    'panel_B_longitudinal_continuous': panel_b_longitudinal_continuous,
    'panel_B_longitudinal_presence': panel_b_longitudinal_presence,
    'panel_C_cohort_continuous': panel_c_cohort_continuous,
    'panel_C_bulbar_continuous': panel_c_bulbar_continuous,
    'panel_C_cohort_presence': panel_c_cohort_presence,
    'panel_C_bulbar_presence': panel_c_bulbar_presence,
}.items():
    export_dataframe(frame, FIGURE_SOURCE_DIR / f'{name}.csv')

print('Main continuous rows:', len(MAIN_CONTINUOUS))
print('Main one-sided rows:', len(MAIN_ONE_SIDED))
print('Sparse/monitoring rows:', len(MAIN_SPARSE))


## 17. Main publication figure: extent, persistence, longitudinal change, cohort, and bulbar structure


In [ ]:
# ------------------------------------------------------------------
# Publication styling
# ------------------------------------------------------------------
mpl.rcParams.update({
    'font.family': ['DejaVu Sans'],
    'font.size': 6.7,
    'axes.titlesize': 8.2,
    'axes.labelsize': 6.9,
    'xtick.labelsize': 6.1,
    'ytick.labelsize': 6.2,
    'legend.fontsize': 6.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

from matplotlib.ticker import FixedLocator, FuncFormatter
from matplotlib.transforms import Bbox

cycle = mpl.rcParams['axes.prop_cycle'].by_key()['color']
CONTROL_COLOR = cycle[0]
ALS_COLOR = cycle[1]
BETWEEN_COLOR = cycle[0]
WITHIN_COLOR = cycle[1]
CORE_COLOR = '0.15'
SUPPORT_COLOR = '0.45'
GRID_COLOR = '0.88'


def ordered_rows(registry_frame: pd.DataFrame) -> tuple[dict[str, float], dict[str, float], dict[str, tuple[float, float]]]:
    positions: dict[str, float] = {}
    centers: dict[str, float] = {}
    bounds: dict[str, tuple[float, float]] = {}
    cursor = 0.0
    for family in ['QADD', 'QGAIN', 'QREV', 'QCHAN']:
        family_frame = registry_frame.loc[
            registry_frame['family'] == family
        ].sort_values('feature_order')
        start = cursor
        for feature in family_frame['feature']:
            positions[feature] = cursor
            cursor += 1.0
        if not family_frame.empty:
            end = cursor - 1.0
            centers[family] = (start + end) / 2.0
            bounds[family] = (start, end)
            cursor += 0.65
    return positions, centers, bounds


def draw_label_axis(
    axis: mpl.axes.Axes,
    registry_frame: pd.DataFrame,
    positions: dict[str, float],
    centers: dict[str, float],
    ylim: tuple[float, float],
) -> None:
    axis.set_xlim(0, 1)
    axis.set_ylim(*ylim)
    axis.axis('off')
    for family, center in centers.items():
        axis.text(
            0.01, center, family,
            ha='left', va='center', fontweight='bold', color='0.38', fontsize=6.4,
        )
    ordered = registry_frame.assign(
        y=registry_frame['feature'].map(positions)
    ).sort_values('y')
    for row in ordered.itertuples(index=False):
        axis.text(
            0.28, row.y, row.short_label,
            ha='left', va='center',
            fontweight='bold' if row.inference_tier == 'core' else 'normal',
            color='0.10', fontsize=6.4,
        )


def add_family_separators(axis: mpl.axes.Axes, bounds: dict[str, tuple[float, float]]) -> None:
    ordered_bounds = list(bounds.values())
    for (_, end), (next_start, _) in zip(ordered_bounds[:-1], ordered_bounds[1:]):
        separator = (end + next_start) / 2.0
        axis.axhline(separator, color='0.90', linewidth=0.55, zorder=0)


def forest(
    axis: mpl.axes.Axes,
    frame: pd.DataFrame,
    positions: dict[str, float],
    *,
    estimate_col: str = 'estimate',
    low_col: str = 'ci95_low',
    high_col: str = 'ci95_high',
    null: float = 0.0,
    log_scale: bool = False,
    color: str = CORE_COLOR,
    star: bool = True,
) -> None:
    axis.axvline(null, color='0.58', linestyle='--', linewidth=0.7, zorder=0)
    for _, row in frame.iterrows():
        feature = row['feature']
        if feature not in positions:
            continue
        estimate = row.get(estimate_col, np.nan)
        low = row.get(low_col, np.nan)
        high = row.get(high_col, np.nan)
        if not np.isfinite(estimate):
            continue
        y = positions[feature]
        tier = row.get('inference_tier', 'secondary')
        marker_face = color if tier == 'core' else 'white'
        if np.isfinite(low) and np.isfinite(high):
            axis.errorbar(
                estimate, y,
                xerr=np.array([[estimate - low], [high - estimate]]),
                fmt='o', color=color, markerfacecolor=marker_face,
                markeredgecolor=color, markersize=3.7, linewidth=0.9,
                capsize=1.7, zorder=3,
            )
            q = row.get('q_value', np.nan)
            if star and np.isfinite(q) and q < 0.05:
                axis.text(high, y, ' *', ha='left', va='center', fontsize=6.0, color=color)
        else:
            axis.scatter(
                estimate, y, s=18, color=marker_face,
                edgecolor=color, linewidth=0.8, zorder=3,
            )
    if log_scale:
        axis.set_xscale('log')
    axis.grid(axis='x', color=GRID_COLOR, linewidth=0.45)
    axis.tick_params(axis='y', length=0, labelleft=False)


def configure_log_or_axis(axis: mpl.axes.Axes, values: pd.Series | np.ndarray) -> None:
    clean = np.asarray(values, dtype=float)
    clean = clean[np.isfinite(clean) & (clean > 0)]
    if clean.size == 0:
        axis.set_xlim(0.5, 2.0)
        ticks = [0.5, 1.0, 2.0]
    else:
        minimum = max(0.25, clean.min() / 1.35)
        maximum = min(8.0, clean.max() * 1.35)
        if minimum >= 1:
            minimum = 0.5
        if maximum <= 1:
            maximum = 2.0
        axis.set_xlim(minimum, maximum)
        candidates = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
        ticks = [tick for tick in candidates if minimum <= tick <= maximum]
        if 1.0 not in ticks:
            ticks.append(1.0)
            ticks = sorted(ticks)
    axis.xaxis.set_major_locator(FixedLocator(ticks))
    axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
    axis.xaxis.set_minor_locator(FixedLocator([]))


def title_axis(axis: mpl.axes.Axes, letter: str, title: str, subtitle: str | None = None) -> None:
    axis.axis('off')
    axis.text(0.0, 0.80, f'{letter}   {title}', ha='left', va='center',
              fontsize=8.15, fontweight='bold')
    if subtitle:
        axis.text(0.0, 0.00, subtitle, ha='left', va='bottom',
                  fontsize=5.55, color='0.42')


def audit_text_overlap(fig: mpl.figure.Figure) -> pd.DataFrame:
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    texts = []
    for artist in fig.findobj(mpl.text.Text):
        text = artist.get_text().strip()
        if not artist.get_visible() or not text:
            continue
        try:
            bbox = artist.get_window_extent(renderer=renderer)
        except Exception:
            continue
        if bbox.width > 0 and bbox.height > 0:
            texts.append((artist, text, bbox))
    records = []
    for i in range(len(texts)):
        artist_a, text_a, bbox_a = texts[i]
        for j in range(i + 1, len(texts)):
            artist_b, text_b, bbox_b = texts[j]
            overlap = Bbox.intersection(bbox_a, bbox_b)
            if overlap is None or overlap.width * overlap.height <= 8:
                continue
            if (
                re.fullmatch(r'(?:−|-|\+)?\d*\.?\d+', text_a)
                and re.fullmatch(r'(?:−|-|\+)?\d*\.?\d+', text_b)
                and overlap.width * overlap.height < 40
            ):
                continue
            # Ignore identical numeric tick labels on separate axes.
            if artist_a.axes is not artist_b.axes:
                if text_a == text_b and re.fullmatch(r'[-+]?\d*\.?\d+', text_a):
                    continue
                # Axes do not overlap in the layout, so separate-axis bounding-box
                # intersections usually arise only at shared boundaries.
                a_box = artist_a.axes.get_window_extent(renderer) if artist_a.axes else None
                b_box = artist_b.axes.get_window_extent(renderer) if artist_b.axes else None
                if a_box is not None and b_box is not None and Bbox.intersection(a_box, b_box) is None:
                    continue
            # Title-axis headline/subtitle pairs are deliberately close; their
            # dedicated axes contain no data or tick labels and are visually audited
            # separately. Do not flag that intended pairing as a data-label collision.
            if artist_a.axes is artist_b.axes and artist_a.axes is not None:
                same_axis_texts = [
                    item.get_text().strip()
                    for item in artist_a.axes.findobj(mpl.text.Text)
                    if item.get_visible() and item.get_text().strip()
                ]
                if len(same_axis_texts) <= 2 and not artist_a.axes.axison:
                    continue
            records.append({
                'text_a': text_a,
                'text_b': text_b,
                'overlap_area_pixels': overlap.width * overlap.height,
                'same_axes': artist_a.axes is artist_b.axes,
            })
    return pd.DataFrame(records)


# ------------------------------------------------------------------
# Canvas
# ------------------------------------------------------------------
fig = plt.figure(figsize=(7.35, 15.4), constrained_layout=False)
outer = fig.add_gridspec(3, 1, height_ratios=[5.25, 4.2, 5.3], hspace=0.44)
fig.suptitle(
    'Recording-quality variation across cohort, time, and bulbar function',
    x=0.52, y=0.995, fontsize=10.2, fontweight='bold',
)

# ==================================================================
# A. Extent and distribution
# ==================================================================
gs_a = outer[0].subgridspec(3, 1, height_ratios=[0.32, 3.75, 1.35], hspace=0.52)
ax_a_title = fig.add_subplot(gs_a[0, 0])
title_axis(
    ax_a_title, 'A', 'Cohort distribution profiles',
    'Thin line: 5th–95th percentile; thick line: interquartile range; point: median.',
)

gs_a_main = gs_a[1, 0].subgridspec(1, 2, width_ratios=[1.55, 4.85], wspace=0.03)
ax_a_labels = fig.add_subplot(gs_a_main[0, 0])
ax_a = fig.add_subplot(gs_a_main[0, 1])
continuous_registry = MAIN_CONTINUOUS.copy()
pos_a, centers_a, bounds_a = ordered_rows(continuous_registry)
ylim_a = (max(pos_a.values()) + 0.55, -0.55)

for row in continuous_registry.itertuples(index=False):
    feature = row.feature
    y = pos_a[feature]
    for cohort, offset, color in [('CONTROL', 0.13, CONTROL_COLOR), ('ALS', -0.13, ALS_COLOR)]:
        source_row = panel_a_standardized.loc[
            (panel_a_standardized['feature'] == feature)
            & (panel_a_standardized['cohort'] == cohort)
        ]
        if source_row.empty:
            continue
        source_row = source_row.iloc[0]
        p05, q1, median, q3, p95 = (
            source_row['p05_z'], source_row['q1_z'], source_row['median_z'],
            source_row['q3_z'], source_row['p95_z'],
        )
        ax_a.plot([p05, p95], [y + offset, y + offset], color=color, linewidth=0.7, alpha=0.78)
        ax_a.plot([q1, q3], [y + offset, y + offset], color=color, linewidth=2.15,
                  solid_capstyle='round')
        ax_a.scatter(median, y + offset, s=17, color=color, edgecolor='white', linewidth=0.4, zorder=3)

ax_a.axvline(0, color='0.60', linestyle='--', linewidth=0.7)
ax_a.set_ylim(*ylim_a)
ax_a.set_xlabel('Frozen transformed and IQR-standardized recording value', labelpad=3)
ax_a.grid(axis='x', color=GRID_COLOR, linewidth=0.45)
ax_a.tick_params(axis='y', left=False, labelleft=False)
add_family_separators(ax_a, bounds_a)
draw_label_axis(ax_a_labels, continuous_registry, pos_a, centers_a, ylim_a)
ax_a.legend(
    handles=[
        mpl.lines.Line2D([0], [0], marker='o', linestyle='-', color=CONTROL_COLOR,
                         label='Control', markersize=4, linewidth=1.1),
        mpl.lines.Line2D([0], [0], marker='o', linestyle='-', color=ALS_COLOR,
                         label='ALS', markersize=4, linewidth=1.1),
    ], frameon=False, loc='upper right', ncol=2,
)

# A2: state and sparse-event prevalence, each with a dedicated label column.
gs_a2 = gs_a[2, 0].subgridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.42)

gs_a_state = gs_a2[0, 0].subgridspec(1, 2, width_ratios=[0.38, 1.0], wspace=0.02)
ax_a_state_labels = fig.add_subplot(gs_a_state[0, 0])
ax_a_state = fig.add_subplot(gs_a_state[0, 1])
state_registry = MAIN_ONE_SIDED.copy()
y_state = np.arange(len(state_registry))[::-1]
for y, row in zip(y_state, state_registry.itertuples(index=False)):
    frame = panel_a_states.loc[panel_a_states['feature'] == row.feature]
    for cohort, marker, color in [('CONTROL', 'o', CONTROL_COLOR), ('ALS', 's', ALS_COLOR)]:
        record = frame.loc[frame['cohort'] == cohort]
        if record.empty:
            continue
        prevalence = 100 - float(record.iloc[0]['zero_percent_available'])
        ax_a_state.scatter(prevalence, y, marker=marker, s=23, color=color,
                           edgecolor='white', linewidth=0.4, zorder=3)
ax_a_state_labels.set_xlim(0, 1)
ax_a_state_labels.set_ylim(-0.65, len(state_registry) - 0.35)
ax_a_state_labels.axis('off')
for y, label, tier in zip(y_state, ['Rolloff-95', 'High-band ratio'], state_registry['inference_tier']):
    ax_a_state_labels.text(
        0.98, y, label, ha='right', va='center', fontsize=6.15,
        fontweight='bold' if tier == 'core' else 'normal',
    )
ax_a_state.set_yticks([])
ax_a_state.set_ylim(-0.65, len(state_registry) - 0.35)
ax_a_state.set_xlim(0, 100)
ax_a_state.set_xlabel('Recordings with a directional deficit (%)', labelpad=3)
ax_a_state.set_title('One-sided QCHAN states', loc='left', fontweight='bold', fontsize=7.1)
ax_a_state.grid(axis='x', color=GRID_COLOR, linewidth=0.45)

gs_a_sparse = gs_a2[0, 1].subgridspec(1, 2, width_ratios=[0.44, 1.0], wspace=0.02)
ax_a_sparse_labels = fig.add_subplot(gs_a_sparse[0, 0])
ax_a_sparse = fig.add_subplot(gs_a_sparse[0, 1])
sparse_registry = MAIN_SPARSE.copy()
y_sparse = np.arange(len(sparse_registry))[::-1]
max_sparse = 0.0
for y, row in zip(y_sparse, sparse_registry.itertuples(index=False)):
    frame = panel_a_sparse.loc[panel_a_sparse['feature'] == row.feature]
    for cohort, marker, color in [('CONTROL', 'o', CONTROL_COLOR), ('ALS', 's', ALS_COLOR)]:
        record = frame.loc[frame['cohort'] == cohort]
        if record.empty:
            continue
        record = record.iloc[0]
        prevalence = (
            100 * float(record['positive_count']) / float(record['n_available_recordings'])
            if record['n_available_recordings'] else np.nan
        )
        if np.isfinite(prevalence):
            max_sparse = max(max_sparse, prevalence)
            ax_a_sparse.scatter(prevalence, y, marker=marker, s=23, color=color,
                                edgecolor='white', linewidth=0.4, zorder=3)
ax_a_sparse_labels.set_xlim(0, 1)
ax_a_sparse_labels.set_ylim(-0.65, len(sparse_registry) - 0.35)
ax_a_sparse_labels.axis('off')
for y, label in zip(y_sparse, ['Hard clipping', 'Dropout-like', 'Frozen audio']):
    ax_a_sparse_labels.text(0.98, y, label, ha='right', va='center', fontsize=6.05)
ax_a_sparse.set_yticks([])
ax_a_sparse.set_ylim(-0.65, len(sparse_registry) - 0.35)
ax_a_sparse.set_xlim(0, max(1.0, max_sparse * 1.35 + 0.05))
ax_a_sparse.set_xlabel('Recordings with a detected event (%)', labelpad=3)
ax_a_sparse.set_title('Sparse QDIST/QTEMP events', loc='left', fontweight='bold', fontsize=7.1)
ax_a_sparse.grid(axis='x', color=GRID_COLOR, linewidth=0.45)

# ==================================================================
# B. Persistence and longitudinal change
# ==================================================================
gs_b = outer[1].subgridspec(2, 1, height_ratios=[0.32, 4.0], hspace=0.12)
ax_b_title = fig.add_subplot(gs_b[0, 0])
title_axis(
    ax_b_title, 'B', 'Participant-level persistence and ALS longitudinal change',
    'ICC quantifies persistent between-participant structure; longitudinal estimates are not interpreted as purely technical drift.',
)
gs_b_plot = gs_b[1, 0].subgridspec(1, 4, width_ratios=[1.62, 1.0, 1.08, 1.0], wspace=0.28)
ax_b_labels = fig.add_subplot(gs_b_plot[0, 0])
ax_b_icc = fig.add_subplot(gs_b_plot[0, 1])
ax_b_slope = fig.add_subplot(gs_b_plot[0, 2])
ax_b_presence = fig.add_subplot(gs_b_plot[0, 3])
all_persistence_registry = pd.concat([MAIN_CONTINUOUS, MAIN_ONE_SIDED]).sort_values(['family', 'feature_order'])
pos_b, centers_b, bounds_b = ordered_rows(all_persistence_registry)
ylim_b = (max(pos_b.values()) + 0.55, -0.55)
draw_label_axis(ax_b_labels, all_persistence_registry, pos_b, centers_b, ylim_b)

icc_frame = panel_b_persistence.rename(columns={
    'icc': 'estimate', 'icc_ci95_low': 'ci95_low', 'icc_ci95_high': 'ci95_high'
})
forest(ax_b_icc, icc_frame, pos_b, null=0, color=CORE_COLOR, star=False)
ax_b_icc.set_ylim(*ylim_b)
ax_b_icc.set_xlim(0, 1)
ax_b_icc.set_xlabel('ICC')
ax_b_icc.set_title('Persistence', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_b_icc, bounds_b)

forest(ax_b_slope, panel_b_longitudinal_continuous, pos_b, null=0, color=CORE_COLOR)
ax_b_slope.set_ylim(*ylim_b)
ax_b_slope.set_xlabel('Change per year\n(IQR units)')
ax_b_slope.set_title('Continuous outputs', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_b_slope, bounds_b)

presence_long = panel_b_longitudinal_presence.copy()
if not presence_long.empty:
    presence_long['estimate_or'] = np.exp(presence_long['estimate'])
    presence_long['low_or'] = np.exp(presence_long['ci95_low'])
    presence_long['high_or'] = np.exp(presence_long['ci95_high'])
forest(
    ax_b_presence, presence_long, pos_b,
    estimate_col='estimate_or', low_col='low_or', high_col='high_or',
    null=1, log_scale=True, color=CORE_COLOR,
)
configure_log_or_axis(
    ax_b_presence,
    pd.concat([
        presence_long.get('low_or', pd.Series(dtype=float)),
        presence_long.get('high_or', pd.Series(dtype=float)),
    ]),
)
ax_b_presence.set_ylim(*ylim_b)
ax_b_presence.set_xlabel('Annual odds ratio')
ax_b_presence.set_title('Positive-state OR', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_b_presence, bounds_b)

# ==================================================================
# C. Cohort and bulbar associations
# ==================================================================
gs_c = outer[2].subgridspec(4, 1, height_ratios=[0.36, 3.42, 0.40, 1.18], hspace=0.32)
ax_c_title = fig.add_subplot(gs_c[0, 0])
title_axis(
    ax_c_title, 'C', 'ALS-control differences and recording-matched bulbar associations',
    'Bulbar effects are displayed per one-point lower score, so positive values indicate a higher raw indicator value with worse bulbar function.',
)

gs_c_cont = gs_c[1, 0].subgridspec(1, 4, width_ratios=[1.62, 1.0, 1.0, 1.0], wspace=0.28)
ax_c_labels = fig.add_subplot(gs_c_cont[0, 0])
ax_c_cohort = fig.add_subplot(gs_c_cont[0, 1])
ax_c_between = fig.add_subplot(gs_c_cont[0, 2])
ax_c_within = fig.add_subplot(gs_c_cont[0, 3])
pos_c, centers_c, bounds_c = ordered_rows(MAIN_CONTINUOUS)
ylim_c = (max(pos_c.values()) + 0.55, -0.55)
draw_label_axis(ax_c_labels, MAIN_CONTINUOUS, pos_c, centers_c, ylim_c)

forest(ax_c_cohort, panel_c_cohort_continuous, pos_c, null=0, color=CORE_COLOR)
ax_c_cohort.set_ylim(*ylim_c)
ax_c_cohort.set_xlabel('IQR-standardized effect')
ax_c_cohort.set_title('Cohort difference', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_c_cohort, bounds_c)

between_frame = panel_c_bulbar_continuous.loc[
    panel_c_bulbar_continuous['target_term'].eq('bulbar_between')
].copy()
within_frame = panel_c_bulbar_continuous.loc[
    panel_c_bulbar_continuous['target_term'].eq('bulbar_within')
].copy()
for frame in [between_frame, within_frame]:
    frame['display_estimate'] = -frame['estimate']
    frame['display_low'] = -frame['ci95_high']
    frame['display_high'] = -frame['ci95_low']

forest(
    ax_c_between, between_frame, pos_c,
    estimate_col='display_estimate', low_col='display_low', high_col='display_high',
    null=0, color=BETWEEN_COLOR,
)
ax_c_between.set_ylim(*ylim_c)
ax_c_between.set_xlabel('Effect per 1-point lower score')
ax_c_between.set_title('Average severity', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_c_between, bounds_c)

forest(
    ax_c_within, within_frame, pos_c,
    estimate_col='display_estimate', low_col='display_low', high_col='display_high',
    null=0, color=WITHIN_COLOR,
)
ax_c_within.set_ylim(*ylim_c)
ax_c_within.set_xlabel('Effect per 1-point lower score')
ax_c_within.set_title('Within-person change', loc='left', fontsize=7.1, fontweight='bold')
add_family_separators(ax_c_within, bounds_c)

# Binary one-sided section with a dedicated heading row.
ax_c_binary_title = fig.add_subplot(gs_c[2, 0])
ax_c_binary_title.axis('off')
ax_c_binary_title.text(
    0.30, 0.05, 'Directional-deficit presence',
    ha='left', va='center', fontsize=7.1, fontweight='bold',
)
gs_c_bin = gs_c[3, 0].subgridspec(1, 4, width_ratios=[1.62, 1.0, 1.0, 1.0], wspace=0.28)
ax_cb_labels = fig.add_subplot(gs_c_bin[0, 0])
ax_cb_cohort = fig.add_subplot(gs_c_bin[0, 1])
ax_cb_between = fig.add_subplot(gs_c_bin[0, 2])
ax_cb_within = fig.add_subplot(gs_c_bin[0, 3])
y_binary = {
    feature: y
    for feature, y in zip(MAIN_ONE_SIDED['feature'], np.arange(len(MAIN_ONE_SIDED))[::-1])
}
ylim_binary = (-0.65, len(MAIN_ONE_SIDED) - 0.35)
draw_label_axis(
    ax_cb_labels, MAIN_ONE_SIDED, y_binary,
    {'QCHAN': np.mean(list(y_binary.values()))}, ylim_binary,
)

cohort_presence = panel_c_cohort_presence.copy()
cohort_presence['or_value'] = np.exp(cohort_presence['estimate'])
cohort_presence['or_low'] = np.exp(cohort_presence['ci95_low'])
cohort_presence['or_high'] = np.exp(cohort_presence['ci95_high'])
forest(
    ax_cb_cohort, cohort_presence, y_binary,
    estimate_col='or_value', low_col='or_low', high_col='or_high',
    null=1, log_scale=True, color=CORE_COLOR,
)
configure_log_or_axis(
    ax_cb_cohort,
    pd.concat([cohort_presence['or_low'], cohort_presence['or_high']]),
)
ax_cb_cohort.set_ylim(*ylim_binary)
ax_cb_cohort.set_xlabel('ALS/control odds ratio')


between_presence = panel_c_bulbar_presence.loc[
    panel_c_bulbar_presence['target_term'].eq('bulbar_between')
].copy()
within_presence = panel_c_bulbar_presence.loc[
    panel_c_bulbar_presence['target_term'].eq('bulbar_within')
].copy()
for frame in [between_presence, within_presence]:
    frame['or_value'] = np.exp(-frame['estimate'])
    frame['or_low'] = np.exp(-frame['ci95_high'])
    frame['or_high'] = np.exp(-frame['ci95_low'])

forest(
    ax_cb_between, between_presence, y_binary,
    estimate_col='or_value', low_col='or_low', high_col='or_high',
    null=1, log_scale=True, color=BETWEEN_COLOR,
)
configure_log_or_axis(
    ax_cb_between,
    pd.concat([between_presence['or_low'], between_presence['or_high']]),
)
ax_cb_between.set_ylim(*ylim_binary)
ax_cb_between.set_xlabel('Between-person OR')

forest(
    ax_cb_within, within_presence, y_binary,
    estimate_col='or_value', low_col='or_low', high_col='or_high',
    null=1, log_scale=True, color=WITHIN_COLOR,
)
configure_log_or_axis(
    ax_cb_within,
    pd.concat([within_presence['or_low'], within_presence['or_high']]),
)
ax_cb_within.set_ylim(*ylim_binary)
ax_cb_within.set_xlabel('Within-person OR')

fig.text(
    0.055, 0.010,
    'Raw indicator directions are retained. † Conditional QREV estimates apply only where the estimator was available. '
    'Filled markers denote core outputs; open markers denote supporting outputs. Asterisks denote q<0.05. '
    'No family-level or global score was formed.',
    fontsize=5.8, ha='left', va='bottom', wrap=True,
)
fig.subplots_adjust(left=0.065, right=0.985, top=0.975, bottom=0.045)

main_paths = save_figure(
    fig, 'Figure_extent_persistence_clinical_structure_PUBLICATION', main=True
)
layout_audit = audit_text_overlap(fig)
export_dataframe(layout_audit, LOG_DIR / 'main_figure_text_overlap_audit.csv')
if not layout_audit.empty:
    display(layout_audit)
    warnings.warn('Potential text overlaps were detected; inspect the audit before final export.')
plt.show()
print(main_paths)


## 18. Supplementary figures: availability, full distributions, trajectories, and sensitivities


In [ ]:
# Supplementary Figure 1: availability and support.
fig, ax = plt.subplots(figsize=(7.2, 8.4))
availability_plot = recording_descriptives.loc[
    recording_descriptives['cohort'].isin(['CONTROL', 'ALS'])
].merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order']], on='feature', how='left'
).sort_values('feature_order')
features = availability_plot['feature'].drop_duplicates().tolist()
y = np.arange(len(features))[::-1]
for y_value, feature in zip(y, features):
    frame = availability_plot.loc[availability_plot['feature'] == feature]
    for cohort, marker, color in [('CONTROL', 'o', CONTROL_COLOR), ('ALS', 's', ALS_COLOR)]:
        row = frame.loc[frame['cohort'] == cohort]
        if row.empty:
            continue
        ax.scatter(float(row.iloc[0]['availability_percent']), y_value,
                   marker=marker, color=color, s=22, edgecolor='white', linewidth=0.4)
ax.set_yticks(y)
ax.set_yticklabels([
    availability_plot.loc[availability_plot['feature'] == feature, 'short_label'].iloc[0]
    for feature in features
])
ax.set_xlim(0, 100)
ax.set_xlabel('Available recordings (%)')
ax.set_title('Supplementary Figure 1 | Measurement availability by cohort', loc='left', fontweight='bold')
ax.grid(axis='x', color=GRID_COLOR, linewidth=0.45)
ax.tick_params(axis='y', length=0)
ax.legend(handles=[
    mpl.lines.Line2D([0], [0], marker='o', linestyle='', color=CONTROL_COLOR, label='Control'),
    mpl.lines.Line2D([0], [0], marker='s', linestyle='', color=ALS_COLOR, label='ALS'),
], frameon=False, loc='lower right')
fig.subplots_adjust(left=0.35, right=0.97, top=0.95, bottom=0.08)
save_figure(fig, 'Supplementary_Figure_1_availability', main=False)
plt.show()

# Supplementary Figure 2: all retained continuous outputs in raw units.
full_continuous = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['outcome_class'].isin(['continuous', 'censored_continuous'])
].sort_values(['family', 'feature_order'])
fig, axes = plt.subplots(2, 2, figsize=(7.2, 9.4), constrained_layout=False)
for axis, family in zip(axes.flat, ['QADD', 'QGAIN', 'QREV', 'QCHAN']):
    family_features = full_continuous.loc[full_continuous['family'] == family]
    y_values = np.arange(len(family_features))[::-1]
    for y_value, row in zip(y_values, family_features.itertuples(index=False)):
        for cohort, offset, color in [('CONTROL', 0.12, CONTROL_COLOR), ('ALS', -0.12, ALS_COLOR)]:
            values = analysis.loc[
                (analysis['cohort'] == cohort) & analysis[f'{row.feature}__available'],
                row.feature,
            ].dropna()
            if values.empty:
                continue
            p05, q1, median, q3, p95 = values.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
            axis.plot([p05, p95], [y_value + offset] * 2, color=color, linewidth=0.7)
            axis.plot([q1, q3], [y_value + offset] * 2, color=color, linewidth=2.0)
            axis.scatter(median, y_value + offset, color=color, s=17, edgecolor='white', linewidth=0.4)
    axis.set_yticks(y_values)
    axis.set_yticklabels(family_features['short_label'])
    axis.set_title(family, loc='left', fontweight='bold')
    axis.grid(axis='x', color=GRID_COLOR, linewidth=0.45)
    axis.tick_params(axis='y', length=0)
fig.suptitle('Supplementary Figure 2 | Raw-unit cohort distribution summaries', fontweight='bold')
fig.subplots_adjust(left=0.22, right=0.97, top=0.94, bottom=0.07, hspace=0.35, wspace=0.45)
save_figure(fig, 'Supplementary_Figure_2_raw_unit_distributions', main=False)
plt.show()

# Supplementary Figure 3: trajectories for core continuous indicators.
trajectory_features = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['inference_tier'].eq('core')
    & FEATURE_REGISTRY['outcome_class'].isin(['continuous', 'censored_continuous'])
].sort_values('feature_order')
fig, axes = plt.subplots(len(trajectory_features), 1, figsize=(7.2, 1.75 * len(trajectory_features)), sharex=True)
axes = np.atleast_1d(axes)
for axis, row in zip(axes, trajectory_features.itertuples(index=False)):
    d = analysis.loc[
        (analysis['cohort'] == 'ALS') & analysis[f'{row.feature}__available']
    ].copy()
    for participant_id, group in d.groupby('participant_id'):
        if len(group) < 2:
            continue
        axis.plot(group['time_since_first_years'], group[f'{row.feature}__z'],
                  color='0.75', linewidth=0.45, alpha=0.55)
    axis.axhline(0, color='0.70', linewidth=0.5, linestyle='--')
    axis.set_ylabel(row.short_label)
    axis.grid(axis='x', color=GRID_COLOR, linewidth=0.4)
axes[-1].set_xlabel('Years since participant first eligible recording')
fig.suptitle('Supplementary Figure 3 | ALS longitudinal trajectories for core indicators', fontweight='bold')
fig.subplots_adjust(left=0.24, right=0.97, top=0.97, bottom=0.04, hspace=0.18)
save_figure(fig, 'Supplementary_Figure_3_longitudinal_trajectories', main=False)
plt.show()

# Supplementary Figure 4: primary and sensitivity cohort estimates.
primary = cohort_models.loc[cohort_models['analysis'].eq('cohort_continuous')].copy()
primary['source'] = 'Primary clustered model'
adjusted = cohort_sensitivity_models.loc[
    cohort_sensitivity_models['analysis'].isin([
        'cohort_demographic_adjusted', 'cohort_untransformed_sensitivity',
        'cohort_winsorized_sensitivity',
    ])
].copy()
adjusted['source'] = adjusted['analysis'].map({
    'cohort_demographic_adjusted': 'Age/sex adjusted',
    'cohort_untransformed_sensitivity': 'Untransformed scale',
    'cohort_winsorized_sensitivity': '1/99% winsorized',
})
sensitivity_plot = pd.concat([primary, adjusted], ignore_index=True).merge(
    FEATURE_REGISTRY[['feature', 'short_label', 'feature_order', 'inference_tier']],
    on='feature', how='left'
)
features = sensitivity_plot['feature'].drop_duplicates().tolist()
y_base = {feature: y for feature, y in zip(features, np.arange(len(features))[::-1])}
offsets = {
    'Primary clustered model': 0.18,
    'Age/sex adjusted': 0.06,
    'Untransformed scale': -0.06,
    '1/99% winsorized': -0.18,
}
fig, ax = plt.subplots(figsize=(7.2, max(4.5, 0.42 * len(features) + 1.6)))
ax.axvline(0, color='0.55', linestyle='--', linewidth=0.7)
for source, frame in sensitivity_plot.groupby('source'):
    for row in frame.itertuples(index=False):
        y_value = y_base[row.feature] + offsets[source]
        ax.errorbar(row.estimate, y_value,
                    xerr=np.array([[row.estimate - row.ci95_low], [row.ci95_high - row.estimate]]),
                    fmt='o', markersize=3.2, linewidth=0.8, capsize=1.5, label=None)
ax.set_yticks([y_base[feature] for feature in features])
ax.set_yticklabels([
    sensitivity_plot.loc[sensitivity_plot['feature'] == feature, 'short_label'].iloc[0]
    for feature in features
])
ax.set_xlabel('ALS − control difference (standardized units)')
ax.set_title('Supplementary Figure 4 | Cohort-model sensitivity analyses', loc='left', fontweight='bold')
ax.grid(axis='x', color=GRID_COLOR, linewidth=0.45)
ax.tick_params(axis='y', length=0)
handles = [mpl.lines.Line2D([0], [0], marker='o', linestyle='', label=source) for source in offsets]
ax.legend(handles=handles, frameon=False, loc='lower right')
fig.subplots_adjust(left=0.32, right=0.97, top=0.94, bottom=0.08)
save_figure(fig, 'Supplementary_Figure_4_cohort_sensitivities', main=False)
plt.show()


## 19. Final manifest, source-data inventory, and completion checks


In [ ]:
# ------------------------------------------------------------------
# Blocking final-run audit
# ------------------------------------------------------------------
eligible_icc_ci = persistence_table.loc[
    persistence_table['icc'].notna()
    & persistence_table[
        'n_repeated_participants'
    ].ge(MIN_REPEATED_PARTICIPANTS)
].copy()

icc_ci_complete = bool(
    not eligible_icc_ci.empty
    and eligible_icc_ci[
        ['icc_ci95_low', 'icc_ci95_high']
    ].notna().all().all()
)

one_recording_counts = (
    one_recording_resampling[
        'n_resamples'
    ].dropna().astype(int)
)

one_recording_complete = bool(
    not one_recording_counts.empty
    and one_recording_counts.eq(
        N_ONE_RECORDING_RESAMPLES
    ).all()
)

demographic_comparison_complete = bool(
    not demographic_adjustment_comparison.empty
    and demographic_adjustment_comparison[
        'n_recordings'
    ].ge(50).all()
)

output_files = sorted(
    path
    for path in OUTPUT_ROOT.rglob('*')
    if path.is_file()
)

manifest = {
    'analysis': FROZEN_SPEC['analysis_name'],
    'created_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'frozen_spec': FROZEN_SPEC,
    'expected_population': {
        'recordings': 519,
        'participants': 224,
    },
    'observed_population': {
        'recordings': int(
            len(analysis)
        ),
        'participants': int(
            analysis[
                'participant_id'
            ].nunique()
        ),
    },
    'paths': {
        'paper_root': str(
            PAPER_ROOT
        ),
        'data_freeze_dir': str(
            DATA_FREEZE_DIR
        ),
        'family_freeze_dir': str(
            FAMILY_FREEZE_DIR
        ),
        'output_root': str(
            OUTPUT_ROOT
        ),
    },
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': scipy.__version__,
        'statsmodels': __import__(
            'statsmodels'
        ).__version__,
        'matplotlib': mpl.__version__,
    },
    'input_files': input_inventory.to_dict(
        orient='records'
    ),
    'output_files': [
        {
            'path': str(path),
            'sha256': sha256_file(path),
            'bytes': path.stat().st_size,
        }
        for path in output_files
    ],
    'final_run_checks': {
        'final_run_enabled': bool(
            FINAL_RUN
        ),
        'requested_icc_bootstrap': int(
            N_ICC_BOOTSTRAP
        ),
        'eligible_icc_rows': int(
            len(eligible_icc_ci)
        ),
        'icc_ci_complete': (
            icc_ci_complete
        ),
        'requested_one_recording_resamples': int(
            N_ONE_RECORDING_RESAMPLES
        ),
        'one_recording_resampling_complete': (
            one_recording_complete
        ),
        'demographic_complete_case_pairs': int(
            len(
                demographic_adjustment_comparison
            )
        ),
    },
}

manifest_path = (
    MANIFEST_DIR
    / 'run_manifest.json'
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding='utf-8',
)

completion = pd.DataFrame(
    [
        {
            'requirement': (
                'Population audit passed'
            ),
            'passed': bool(
                population_audit[
                    'passed'
                ].all()
            ),
            'detail': (
                f"recordings={len(analysis)}; "
                f"participants="
                f"{analysis['participant_id'].nunique()}"
            ),
        },
        {
            'requirement': (
                'Final run enabled'
            ),
            'passed': bool(
                FINAL_RUN
            ),
            'detail': (
                f'FINAL_RUN={FINAL_RUN}'
            ),
        },
        {
            'requirement': (
                'Feature registry exported'
            ),
            'passed': (
                TABLE_DIR
                / 'feature_registry.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Recording descriptives exported'
            ),
            'passed': (
                TABLE_DIR
                / 'recording_level_descriptives.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Persistence exported'
            ),
            'passed': (
                TABLE_DIR
                / 'participant_level_persistence.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'ICC bootstrap confidence intervals complete'
            ),
            'passed': (
                icc_ci_complete
            ),
            'detail': (
                f'eligible_rows={len(eligible_icc_ci)}; '
                f'bootstrap_replicates={N_ICC_BOOTSTRAP}'
            ),
        },
        {
            'requirement': (
                'Longitudinal models exported'
            ),
            'passed': (
                TABLE_DIR
                / 'als_longitudinal_models.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Cohort models exported'
            ),
            'passed': (
                TABLE_DIR
                / 'cohort_models.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Demographic complete-case comparison exported'
            ),
            'passed': (
                demographic_comparison_complete
                and (
                    TABLE_DIR
                    / 'cohort_demographic_adjustment_comparison.csv'
                ).exists()
            ),
            'detail': (
                f'pairs='
                f'{len(demographic_adjustment_comparison)}'
            ),
        },
        {
            'requirement': (
                'Bulbar models exported'
            ),
            'passed': (
                TABLE_DIR
                / 'bulbar_models.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Participant-collapsed sensitivity exported'
            ),
            'passed': (
                TABLE_DIR
                / 'participant_collapsed_cohort_sensitivity.csv'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'One-recording resampling complete'
            ),
            'passed': (
                one_recording_complete
            ),
            'detail': (
                f'resamples='
                f'{N_ONE_RECORDING_RESAMPLES}'
            ),
        },
        {
            'requirement': (
                'Main SVG exported'
            ),
            'passed': (
                FIGURE_MAIN_DIR
                / 'Figure_extent_persistence_clinical_structure_PUBLICATION.svg'
            ).exists(),
            'detail': '',
        },
        {
            'requirement': (
                'Figure source data exported'
            ),
            'passed': (
                len(
                    list(
                        FIGURE_SOURCE_DIR.glob(
                            '*.csv'
                        )
                    )
                )
                >= 10
            ),
            'detail': '',
        },
        {
            'requirement': (
                'Run manifest exported'
            ),
            'passed': (
                manifest_path.exists()
            ),
            'detail': '',
        },
    ]
)

display(completion)

export_dataframe(
    completion,
    LOG_DIR / 'completion_checklist.csv',
)

if not completion[
    'passed'
].all():
    raise AssertionError(
        'The final production notebook did not complete all required exports.'
    )

print(
    'FINAL GOAL 2 RUN COMPLETED. '
    f'Outputs: {OUTPUT_ROOT}'
)